# Road Damage Detection - Google Colab training notebook

Trains the road-damage YOLO detector on a free Colab GPU. All project code is embedded in this
notebook; the only thing you provide is the RDD2022 India dataset.

**Before running**
1. Choose how the notebook gets the RDD2022 India dataset (set `DATASET` in the settings cell):
   - **a download link** (fastest, uses no Drive space): paste the link you downloaded the zip from;
   - **a zip uploaded into Colab**: Files panel (folder icon, left) → upload `RDD2022_India.zip`
     → `DATASET = "/content/RDD2022_India.zip"` (must be re-uploaded after a disconnect);
   - **Google Drive**: *My Drive → road-damage → RDD2022_India.zip* or the extracted `RDD2022_India/` folder.
2. Optional - your own labelled images (XML file next to each image, made with `src/annotate_boxes.py`):
   zip the folder, upload it to *My Drive → road-damage*, and set `EXTRA_DATASET`. They are used for
   **training only**; validation and test stay pure RDD2022 India.
3. **Runtime → Change runtime type → T4 GPU → Save.**
4. **Runtime → Run all**, and allow access when Colab asks to connect to Google Drive.

Everything important is saved to *My Drive → road-damage → outputs* while training runs:
`models/road_damage_yolo.pt` (the trained model), `reports/` and `runs/`.

**If the session disconnects:** reconnect, set `RESUME = True` in the settings cell and
*Run all* again - training continues from the last saved epoch.

Generated by `tools/build_colab_notebook.py` - edit the project files, not this notebook's code cells.

In [ ]:
# ===================== SETTINGS - edit these if needed =====================
DRIVE_DIR = "/content/drive/MyDrive/road-damage"          # your folder in Google Drive
# Where the RDD2022 India dataset comes from - keep exactly ONE of these lines active:
DATASET = "PASTE_DOWNLOAD_LINK_HERE"                      # a direct download link to the zip
# DATASET = "/content/RDD2022_India.zip"                  # a zip uploaded via Colab's Files panel
# DATASET = f"{DRIVE_DIR}/RDD2022_India.zip"              # a zip in Google Drive
# DATASET = f"{DRIVE_DIR}/RDD2022_India"                  # an extracted folder in Google Drive
# Your own labelled images (zip or folder; XML next to each image). None = not used.
EXTRA_DATASET = None
# EXTRA_DATASET = f"{DRIVE_DIR}/road_damage_assesment.zip"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"   # models/reports/runs saved here (~50 MB free Drive space needed)

QUICK_TEST = False   # True = train 1 epoch only (checks everything works in ~5 minutes)
RESUME = False       # True = continue the last interrupted training run (after a disconnect)
MODEL = None         # None = config.yaml default (yolo11n.pt); or "yolo11s.pt"
EPOCHS = None        # None = config.yaml default (100)
IMGSZ = None         # None = config.yaml default (640)
BATCH = None         # None = config.yaml default (16); use 8 if you get CUDA out of memory
# ===========================================================================

## 0. Connect Google Drive and check the GPU

In [ ]:
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab - skipping Google Drive mount.")

PROJ = Path("/content/proj")
RAW_DIR = Path("/content/rdd_raw")
EXTRA_DIR = Path("/content/extra_raw")
(PROJ / "src").mkdir(parents=True, exist_ok=True)

def run(args):
    """Run a command inside the project folder, stream its output, stop on failure."""
    print(">>", " ".join(map(str, args)), flush=True)
    proc = subprocess.Popen([str(a) for a in args], cwd=PROJ, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, encoding="utf-8",
                            errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {proc.returncode}): {' '.join(map(str, args))}")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

IS_URL = DATASET.startswith(("http://", "https://"))
if DATASET.startswith("PASTE_"):
    raise ValueError("Set DATASET in the settings cell (download link, uploaded zip or Drive path).")
if not IS_URL and not Path(DATASET).exists():
    raise FileNotFoundError(f"{DATASET} not found. Check DATASET in the settings cell.")
print("Dataset source:", DATASET)
if EXTRA_DATASET and not Path(EXTRA_DATASET).exists():
    raise FileNotFoundError(f"{EXTRA_DATASET} not found. Check EXTRA_DATASET in the settings cell.")
print("Extra training images:", EXTRA_DATASET or "none")

## Project code
These cells write the project files into `/content/proj`.

In [ ]:
%%writefile /content/proj/config.yaml
# Road Damage Detection and Severity Assessment - project configuration.
# All relative paths are resolved against the project root (the folder that
# contains this file), not the current working directory.

paths:
  reports_dir: reports
  runs_dir: runs/detect
  models_dir: models

dataset:
  # Put the extracted RDD2022 India download anywhere under this folder.
  # Images and XML files are discovered recursively.
  # Currently: your own category dataset (XML made by src/annotate_boxes.py).
  # To use RDD2022 India again: raw_dir: data/raw/RDD2022_India
  raw_dir: data/raw/road_damage_assesment
  # Intermediate output of convert_voc_to_yolo.py (labels + manifest).
  interim_dir: data/interim
  # Final Ultralytics-style dataset produced by split_dataset.py.
  yolo_dir: data/yolo
  image_extensions: [".jpg", ".jpeg", ".png", ".webp"]
  annotation_extension: ".xml"
  # Your own labelled images (Pascal VOC XML next to each image, e.g. made with
  # src/annotate_boxes.py). They are added to the TRAINING split only, so
  # validation and test stay pure RDD2022 India. Use [] to disable.
  extra_dirs: []
  # Only images that have a matching XML file are used. Images without XML
  # (e.g. the unlabeled RDD2022 test folder) are never treated as negatives.
  # Boxes whose clipped width or height is below this many pixels are dropped
  # (and reported).
  min_box_size_px: 1.0

classes:
  # YOLO class id -> class name used in data.yaml and in the UI.
  names:
    0: longitudinal_crack
    1: transverse_crack
    2: alligator_crack
    3: pothole
  # RDD2022 label code -> YOLO class id. Any other label code is ignored
  # (and counted in the conversion summary).
  source_codes:
    D00: 0
    D10: 1
    D20: 2
    D40: 3

model:
  # Pretrained Ultralytics checkpoint used as the starting point for training.
  # Examples: yolo11n.pt (fastest), yolo11s.pt (more accurate).
  name: yolo11n.pt
  confidence: 0.25
  iou: 0.45

training:
  image_size: 640
  epochs: 100
  batch: 16
  workers: 4
  patience: 20
  seed: 42
  device: auto          # auto | cpu | 0 | 0,1 | cuda:0
  deterministic: true
  run_name: rdd2022_india
  # After training, best.pt is copied here for evaluation and the app.
  export_weights_to: models/road_damage_yolo.pt

split:
  train: 0.70
  val: 0.15
  test: 0.15
  seed: 42
  # How related images are kept in the same split to reduce leakage:
  #   random - independent per-image random split (default; RDD2022 India
  #            filenames carry no reliable sequence/location information)
  #   block  - group images by consecutive numeric id in the filename
  #            (e.g. India_000123 -> block 123 // block_size). A heuristic to
  #            keep neighbouring frames together.
  #   regex  - group by the first capture group of group_regex applied to the
  #            image filename stem.
  strategy: regex
  block_size: 50
  # Keeps edited copies of one photo together (Roboflow names like 644_jpg.rf.<hash> -> group 644);
  # other file names form their own group, i.e. a normal random split.
  group_regex: '(\d+)_(?:jpg|jpeg|png)\.rf\.'
  # copy | hardlink | symlink. Links save disk space; symlinks on Windows need
  # Developer Mode or admin rights. Falls back to copy if linking fails.
  link_mode: copy

evaluation:
  split: test
  # Ultralytics evaluates mAP with a very low confidence threshold; these are
  # the standard values and are kept separate from the UI thresholds.
  confidence: 0.001
  iou: 0.6
  num_example_images: 12

inference:
  # Weights used by evaluate_detector.py, crop_defects.py and the app.
  weights: models/road_damage_yolo.pt
  box_thickness: 2
  font_scale: 0.6

severity:
  # auto      - trained classifier if model_path exists, otherwise the heuristic estimate
  # model     - trained classifier only ("Not available" if missing)
  # heuristic - rule-based estimate only
  # none      - no severity
  method: auto
  # Written by src/train_severity.py (see src/severity.py for the format).
  model_path: models/severity_classifier.pt
  crops_dir: data/severity_crops
  # Relative padding added around each box when cropping (0.1 = 10%).
  crop_padding: 0.10
  min_crop_size_px: 8

  # Rule-based ESTIMATE used until a trained classifier exists. These are
  # documented starting values, not values learned from ground truth;
  # train_severity.py reports how well they agree with your labels.
  heuristic:
    # Pixels this many grey levels darker than the crop median count as damage.
    dark_delta: 25
    # Size (fraction of the image) that counts as maximal: crack length for
    # linear classes, sqrt(area fraction) for area defects.
    size_ref:
      linear: 0.6
      area: 0.4
    # Dark-pixel fraction that counts as maximal density.
    density_ref: 0.25
    # score = size_weight * size + (1 - size_weight) * density
    size_weight: 0.6
    # score < 0.33 -> Low, < 0.66 -> Medium, otherwise High
    thresholds: [0.33, 0.66]
    linear_classes: [longitudinal_crack, transverse_crack]

  # Severity classifier training (src/train_severity.py).
  training:
    labels_file: data/severity_crops/labels.csv
    arch: mobilenet_v3_small
    image_size: 224
    epochs: 30
    batch: 32
    learning_rate: 0.001
    weight_decay: 0.0001
    patience: 8
    val_fraction: 0.2
    min_per_level: 5
    seed: 42


In [ ]:
%%writefile /content/proj/src/__init__.py
"""Road Damage Detection and Severity Assessment package."""


In [ ]:
%%writefile /content/proj/src/annotate_boxes.py
"""Create Pascal VOC XML files for extra images sorted into class folders.

Two modes:

* --whole-image : one box covering the entire image (only valid when the defect
                  fills the photo, e.g. top-down close-ups of alligator cracking).
* interactive   : an OpenCV window where you draw a box around each defect.

XML files are written next to each image (same name, .xml), in the same format
as RDD2022, so convert_voc_to_yolo.py reads them unchanged. Images that are too
small, or exact copies of an image already handled, are skipped automatically.
Images you skip (diagrams, collages, no visible defect) get no XML and are
therefore not used. Progress is kept in <folder>/annotation_log.csv, so you can
stop at any time and continue later.

Usage:
    python src/annotate_boxes.py --folder data/raw/extra/alligator_cracks --class alligator_crack --whole-image
    python src/annotate_boxes.py --folder data/raw/extra/potholes --class pothole

Keys in the interactive window:
    mouse drag   draw a box (class = current class, shown at the top)
    1 2 3 4      current class: 1 longitudinal, 2 transverse, 3 alligator, 4 pothole
    u            undo the last box
    Enter/Space  save boxes and go to the next image
    s            skip image (not usable: diagram, collage, no defect, wrong class...)
    b            back to the previous image
    q / Esc      save progress and quit
"""

from __future__ import annotations

import argparse
import hashlib
import logging
import sys
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

from src.common import get_class_names, get_image_extensions, get_source_code_map, load_config, resolve_path, setup_logging
from src.inference import CLASS_COLORS_BGR

logger = logging.getLogger("annotate_boxes")

LOG_NAME = "annotation_log.csv"
LOG_COLUMNS = ["file", "status", "boxes", "note", "md5", "updated"]
MIN_IMAGE_SIDE_PX = 100      # smaller images are skipped automatically
MIN_BOX_SIDE_PX = 4          # ignore accidental clicks
WINDOW = "annotate - drag boxes | 1-4 class | u undo | Enter next | s skip | b back | q quit"
MAX_DISPLAY_SIDE = 1000      # large photos are shown scaled down
KEY_ENTER = (13, 10, 32)
KEY_ESC = 27


def load_image_upright(path: Path) -> np.ndarray | None:
    """Load as BGR with EXIF rotation applied (the orientation YOLO trains on)."""
    try:
        with Image.open(path) as img:
            rgb = np.asarray(ImageOps.exif_transpose(img).convert("RGB"))
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    except OSError:
        return None


def file_md5(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()


def xml_path_for(image_path: Path) -> Path:
    """<stem>.xml, or <file name>.xml if another image in the folder has the same stem."""
    clash = any(p != image_path and p.stem == image_path.stem and p.suffix.lower() != ".xml"
                for p in image_path.parent.glob(f"{glob_escape(image_path.stem)}.*"))
    return image_path.with_name(image_path.name + ".xml") if clash else image_path.with_suffix(".xml")


def glob_escape(text: str) -> str:
    return "".join(f"[{c}]" if c in "[]*?" else c for c in text)


def remove_xml(image_path: Path) -> None:
    for candidate in (image_path.with_suffix(".xml"), image_path.with_name(image_path.name + ".xml")):
        if candidate.is_file() and candidate == xml_path_for(image_path):
            candidate.unlink()


def write_voc_xml(image_path: Path, width: int, height: int, boxes: list[tuple[str, list[int]]]) -> Path:
    """Write <image>.xml next to the image; boxes are (code, [x1, y1, x2, y2])."""
    root = ET.Element("annotation")
    ET.SubElement(root, "folder").text = image_path.parent.name
    ET.SubElement(root, "filename").text = image_path.name
    size = ET.SubElement(root, "size")
    for tag, value in (("width", width), ("height", height), ("depth", 3)):
        ET.SubElement(size, tag).text = str(value)
    for code, (x1, y1, x2, y2) in boxes:
        obj = ET.SubElement(root, "object")
        ET.SubElement(obj, "name").text = code
        ET.SubElement(obj, "difficult").text = "0"
        bnd = ET.SubElement(obj, "bndbox")
        for tag, value in (("xmin", x1), ("ymin", y1), ("xmax", x2), ("ymax", y2)):
            ET.SubElement(bnd, tag).text = str(int(value))
    ET.indent(root)
    xml_path = xml_path_for(image_path)
    ET.ElementTree(root).write(xml_path, encoding="utf-8", xml_declaration=True)
    return xml_path


def read_voc_boxes(xml_path: Path) -> list[tuple[str, list[int]]]:
    """Boxes of an existing XML (to show them again when going back)."""
    if not xml_path.is_file():
        return []
    boxes = []
    for obj in ET.parse(xml_path).getroot().findall("object"):
        b = obj.find("bndbox")
        boxes.append((obj.findtext("name", ""), [int(float(b.findtext(t))) for t in ("xmin", "ymin", "xmax", "ymax")]))
    return boxes


class AnnotationLog:
    """annotation_log.csv: one row per image with its status."""

    def __init__(self, folder: Path) -> None:
        self.path = folder / LOG_NAME
        self.df = pd.read_csv(self.path, dtype=str).fillna("") if self.path.is_file() else pd.DataFrame(columns=LOG_COLUMNS)

    def status(self, name: str) -> str | None:
        rows = self.df[self.df["file"] == name]
        return rows["status"].iloc[0] if len(rows) else None

    def set(self, name: str, status: str, boxes: int = 0, note: str = "", md5: str = "") -> None:
        self.df = self.df[self.df["file"] != name]
        row = {"file": name, "status": status, "boxes": boxes, "note": note, "md5": md5,
               "updated": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self.df = pd.concat([self.df, pd.DataFrame([row])], ignore_index=True)
        self.df.to_csv(self.path, index=False)


def known_hashes(extra_root: Path, current_folder: Path) -> dict[str, str]:
    """md5 -> 'folder/file' of images already labeled or skipped in any class folder."""
    hashes: dict[str, str] = {}
    for log_path in extra_root.rglob(LOG_NAME):
        df = pd.read_csv(log_path, dtype=str).fillna("")
        for row in df.itertuples(index=False):
            if row.md5 and row.status in ("labeled", "whole_image"):
                hashes.setdefault(row.md5, f"{log_path.parent.name}/{row.file}")
    return hashes


def other_folder_hashes(folder: Path, extensions: set[str]) -> dict[str, str]:
    """md5 -> 'folder/file' for images in the sibling class folders."""
    hashes: dict[str, str] = {}
    for sibling in folder.parent.iterdir():
        if sibling.is_dir() and sibling != folder:
            for f in sibling.iterdir():
                if f.is_file() and f.suffix.lower() in extensions:
                    hashes.setdefault(file_md5(f), f"{sibling.name}/{f.name}")
    return hashes


def auto_skip_reason(path: Path, image: np.ndarray | None, md5: str, seen: dict[str, str]) -> str | None:
    if image is None:
        return "unreadable"
    if min(image.shape[:2]) < MIN_IMAGE_SIDE_PX:
        return f"too small ({image.shape[1]}x{image.shape[0]})"
    if md5 in seen and seen[md5] != f"{path.parent.name}/{path.name}":
        return f"duplicate of {seen[md5]}"
    return None


def run_whole_image(images: list[Path], code: str, log: AnnotationLog, seen: dict[str, str],
                    in_other_folders: dict[str, str]) -> None:
    """One full-image box per image. Images that also appear in another class folder are
    ambiguous (the folder cannot tell which class is right) and are skipped."""
    done = skipped = 0
    for path in images:
        image = load_image_upright(path)
        md5 = file_md5(path)
        reason = auto_skip_reason(path, image, md5, seen)
        if not reason and md5 in in_other_folders:
            reason = f"ambiguous: same image is also in {in_other_folders[md5]}"
        if not reason and md5 in seen and seen[md5] != f"{path.parent.name}/{path.name}":
            reason = f"duplicate of {seen[md5]}"
        if reason:
            remove_xml(path)
            log.set(path.name, "auto_skipped", note=reason, md5=md5)
            skipped += 1
            continue
        h, w = image.shape[:2]
        write_voc_xml(path, w, h, [(code, [0, 0, w, h])])
        log.set(path.name, "whole_image", boxes=1, md5=md5)
        seen[md5] = f"{path.parent.name}/{path.name}"
        done += 1
    logger.info("Whole-image XML written for %d images, %d skipped automatically.", done, skipped)


class BoxDrawer:
    """Interactive OpenCV box drawing for one image."""

    def __init__(self, image: np.ndarray, boxes: list[tuple[str, list[int]]], code: str,
                 code_by_key: dict[int, str], names_by_code: dict[str, str], title: str) -> None:
        self.image = image
        self.scale = min(1.0, MAX_DISPLAY_SIDE / max(image.shape[:2]))
        self.boxes = list(boxes)
        self.code = code
        self.code_by_key = code_by_key
        self.names_by_code = names_by_code
        self.title = title
        self.start: tuple[int, int] | None = None
        self.current: tuple[int, int] | None = None

    def on_mouse(self, event: int, x: int, y: int, flags: int, param: Any) -> None:
        if event == cv2.EVENT_LBUTTONDOWN:
            self.start, self.current = (x, y), (x, y)
        elif event == cv2.EVENT_MOUSEMOVE and self.start:
            self.current = (x, y)
        elif event == cv2.EVENT_LBUTTONUP and self.start:
            (sx, sy), inv = self.start, 1.0 / self.scale
            h, w = self.image.shape[:2]
            x1, x2 = sorted((int(sx * inv), int(x * inv)))
            y1, y2 = sorted((int(sy * inv), int(y * inv)))
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
            if x2 - x1 >= MIN_BOX_SIDE_PX and y2 - y1 >= MIN_BOX_SIDE_PX:
                self.boxes.append((self.code, [x1, y1, x2, y2]))
            self.start = self.current = None

    def render(self) -> np.ndarray:
        canvas = cv2.resize(self.image, None, fx=self.scale, fy=self.scale) if self.scale < 1 else self.image.copy()
        class_ids = {c: i for i, c in enumerate(self.names_by_code)}
        for code, (x1, y1, x2, y2) in self.boxes:
            color = CLASS_COLORS_BGR.get(class_ids.get(code, -1), (200, 200, 200))
            p1, p2 = (int(x1 * self.scale), int(y1 * self.scale)), (int(x2 * self.scale), int(y2 * self.scale))
            cv2.rectangle(canvas, p1, p2, color, 2)
            cv2.putText(canvas, self.names_by_code.get(code, code), (p1[0] + 3, p1[1] + 16),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)
        if self.start and self.current:
            cv2.rectangle(canvas, self.start, self.current, (0, 255, 255), 1)
        keys = "  ".join(f"{chr(k)}={self.names_by_code[c]}" for k, c in self.code_by_key.items())
        header = f"{self.title} | drawing: {self.names_by_code[self.code]} | {keys}"
        cv2.rectangle(canvas, (0, 0), (canvas.shape[1], 24), (0, 0, 0), cv2.FILLED)
        cv2.putText(canvas, header, (6, 17), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        return canvas

    def run(self) -> str:
        """Returns 'next', 'skip', 'back' or 'quit'."""
        cv2.setMouseCallback(WINDOW, self.on_mouse)
        while True:
            cv2.imshow(WINDOW, self.render())
            key = cv2.waitKey(20) & 0xFF
            if key == 255:
                if cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE) < 1:
                    return "quit"
                continue
            if key in KEY_ENTER:
                return "next"
            if key in (ord("q"), KEY_ESC):
                return "quit"
            if key == ord("s"):
                return "skip"
            if key == ord("b"):
                return "back"
            if key == ord("u") and self.boxes:
                self.boxes.pop()
            if key in self.code_by_key:
                self.code = self.code_by_key[key]


def run_interactive(images: list[Path], code: str, log: AnnotationLog, seen: dict[str, str],
                    code_by_key: dict[int, str], names_by_code: dict[str, str], redo: bool) -> None:
    todo = [p for p in images if redo or log.status(p.name) is None]
    logger.info("%d of %d images still to annotate in this folder.", len(todo), len(images))
    cv2.namedWindow(WINDOW, cv2.WINDOW_AUTOSIZE)
    i = 0
    while 0 <= i < len(todo):
        path = todo[i]
        image = load_image_upright(path)
        md5 = file_md5(path)
        reason = auto_skip_reason(path, image, md5, seen)
        if reason:
            log.set(path.name, "auto_skipped", note=reason, md5=md5)
            logger.info("Auto-skipped %s: %s", path.name, reason)
            i += 1
            continue
        drawer = BoxDrawer(image, read_voc_boxes(xml_path_for(path)), code, code_by_key, names_by_code,
                           f"[{i + 1}/{len(todo)}] {path.name}")
        action = drawer.run()
        if action == "quit":
            break
        if action == "back":
            i = max(0, i - 1)
            continue
        if action == "skip" or not drawer.boxes:
            remove_xml(path)
            log.set(path.name, "skipped", note="skipped by annotator" if action == "skip" else "no boxes", md5=md5)
        else:
            h, w = image.shape[:2]
            write_voc_xml(path, w, h, drawer.boxes)
            log.set(path.name, "labeled", boxes=len(drawer.boxes), md5=md5)
            seen[md5] = f"{path.parent.name}/{path.name}"
        i += 1
    cv2.destroyAllWindows()


def summarize(folder: Path, log: AnnotationLog) -> None:
    counts = log.df["status"].value_counts().to_dict()
    boxes = pd.to_numeric(log.df["boxes"], errors="coerce").fillna(0).sum()
    logger.info("%s: %s | %d boxes in total. Log: %s", folder.name, counts, int(boxes), log.path)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--folder", required=True, help="Folder with images of one class")
    parser.add_argument("--class", dest="class_name", required=True,
                        help="Default class: longitudinal_crack | transverse_crack | alligator_crack | pothole")
    parser.add_argument("--whole-image", action="store_true", help="One box = whole image (close-ups only)")
    parser.add_argument("--redo", action="store_true", help="Revisit images that were already handled")
    args = parser.parse_args()

    setup_logging()
    cfg = load_config(args.config)
    names = get_class_names(cfg)
    code_of = {names[cid]: code for code, cid in get_source_code_map(cfg).items()}
    if args.class_name not in code_of:
        logger.error("Unknown class '%s'. Use one of: %s", args.class_name, ", ".join(code_of))
        return 1
    names_by_code = {code_of[names[cid]]: names[cid] for cid in sorted(names)}
    code_by_key = {ord(str(i + 1)): code for i, code in enumerate(names_by_code)}

    folder = resolve_path(args.folder)
    if not folder or not folder.is_dir():
        logger.error("Folder not found: %s", args.folder)
        return 1
    images = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in get_image_extensions(cfg))
    if not images:
        logger.error("No images in %s", folder)
        return 1
    log = AnnotationLog(folder)
    seen = known_hashes(folder.parent, folder)
    if args.whole_image:
        run_whole_image(images, code_of[args.class_name], log, seen,
                        other_folder_hashes(folder, get_image_extensions(cfg)))
    else:
        run_interactive(images, code_of[args.class_name], log, seen, code_by_key, names_by_code, args.redo)
    summarize(folder, log)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/common.py
"""Shared helpers: configuration, paths, logging, seeding and device selection."""

from __future__ import annotations

import logging
import os
import random
from pathlib import Path
from typing import Any

import numpy as np
import yaml

PROJECT_ROOT = Path(__file__).resolve().parents[1]
DEFAULT_CONFIG_PATH = PROJECT_ROOT / "config.yaml"

REQUIRED_SECTIONS = ("paths", "dataset", "classes", "model", "training", "split", "inference", "severity")

logger = logging.getLogger(__name__)


class ConfigError(RuntimeError):
    """Raised when config.yaml is missing or invalid."""


def setup_logging(level: int = logging.INFO) -> None:
    """Configure a simple console logger for command-line scripts."""
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)-7s | %(message)s",
        datefmt="%H:%M:%S",
    )


def load_config(path: str | Path | None = None) -> dict[str, Any]:
    """Load and minimally validate the YAML configuration file."""
    config_path = Path(path) if path else DEFAULT_CONFIG_PATH
    if not config_path.is_file():
        raise ConfigError(f"Config file not found: {config_path}")
    with config_path.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    missing = [s for s in REQUIRED_SECTIONS if s not in cfg]
    if missing:
        raise ConfigError(f"Config {config_path} is missing sections: {', '.join(missing)}")
    return cfg


def resolve_path(path: str | Path | None) -> Path | None:
    """Resolve a config path relative to the project root (absolute paths are kept)."""
    if path is None or str(path).strip() == "":
        return None
    p = Path(path).expanduser()
    return p if p.is_absolute() else (PROJECT_ROOT / p).resolve()


def to_project_relative(path: Path) -> str:
    """Forward-slash path relative to the project root if inside it, else absolute (for CSV files)."""
    resolved = Path(path).resolve()
    try:
        return resolved.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return resolved.as_posix()


def get_class_names(cfg: dict[str, Any]) -> dict[int, str]:
    """Return {class_id: class_name} from the config."""
    return {int(k): str(v) for k, v in cfg["classes"]["names"].items()}


def get_source_code_map(cfg: dict[str, Any]) -> dict[str, int]:
    """Return {RDD2022 label code: class_id} from the config."""
    names = get_class_names(cfg)
    mapping = {str(k).strip().upper(): int(v) for k, v in cfg["classes"]["source_codes"].items()}
    unknown = [cid for cid in mapping.values() if cid not in names]
    if unknown:
        raise ConfigError(f"classes.source_codes references unknown class ids: {unknown}")
    return mapping


def get_image_extensions(cfg: dict[str, Any]) -> set[str]:
    """Return the lower-case image extensions (with dot) to discover."""
    return {e.lower() if e.startswith(".") else f".{e.lower()}" for e in cfg["dataset"]["image_extensions"]}


def pretty_class_name(name: str) -> str:
    """'longitudinal_crack' -> 'Longitudinal crack'."""
    return name.replace("_", " ").strip().capitalize()


def set_seed(seed: int) -> None:
    """Seed Python, NumPy and (if installed) PyTorch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        logger.debug("PyTorch not installed; skipping torch seeding.")


def cuda_available() -> bool:
    """Return True if PyTorch is installed and can see a CUDA GPU."""
    try:
        import torch

        return bool(torch.cuda.is_available())
    except ImportError:
        return False


def resolve_device(device: str | int | None) -> str:
    """Translate 'auto' into '0' (first GPU) or 'cpu'; pass explicit values through.

    Falls back to CPU with a warning if a GPU was requested but CUDA is unavailable.
    """
    value = "auto" if device is None else str(device).strip().lower()
    has_cuda = cuda_available()
    if value in ("", "auto"):
        return "0" if has_cuda else "cpu"
    if value != "cpu" and not has_cuda:
        logger.warning("Device '%s' requested but CUDA is not available; using CPU.", device)
        return "cpu"
    return value.replace("cuda:", "")


def find_latest_weights(runs_dir: Path) -> Path | None:
    """Return the most recently modified runs/detect/*/weights/best.pt, if any."""
    if not runs_dir.is_dir():
        return None
    candidates = sorted(runs_dir.glob("*/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def resolve_weights(cfg: dict[str, Any], weights: str | Path | None = None) -> Path:
    """Find detector weights: explicit arg > config inference.weights > latest training run.

    Raises FileNotFoundError with a helpful message if nothing is found.
    """
    if weights:
        path = resolve_path(weights)
        if path and path.is_file():
            return path
        raise FileNotFoundError(f"Weights file not found: {path}")
    configured = resolve_path(cfg["inference"].get("weights"))
    if configured and configured.is_file():
        return configured
    latest = find_latest_weights(resolve_path(cfg["paths"]["runs_dir"]))
    if latest:
        logger.info("Configured weights not found; using latest training run: %s", latest)
        return latest
    raise FileNotFoundError(
        "No trained detector found. Train one with `python src/train_detector.py` "
        f"(expected {configured} or runs/detect/*/weights/best.pt)."
    )


In [ ]:
%%writefile /content/proj/src/convert_voc_to_yolo.py
"""Convert RDD2022 Pascal VOC XML annotations to YOLO label files.

For every image with a matching XML file, writes ``<interim_dir>/labels/<uid>.txt``
containing zero or more lines ``class_id cx cy w h`` (normalised to [0, 1]) and
records the pair in ``<interim_dir>/manifest.csv`` for split_dataset.py.

Folders listed in ``dataset.extra_dirs`` (your own labelled images) are converted
too and marked ``source=extra`` in the manifest; split_dataset.py puts them in
the training split only, so validation/test remain pure RDD2022 India.

Usage:
    python src/convert_voc_to_yolo.py [--config config.yaml] [--force] [--no-extra]
"""

from __future__ import annotations

import argparse
import logging
import shutil
import sys
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import pandas as pd
from tqdm import tqdm

from src.common import (
    get_class_names,
    get_image_extensions,
    get_source_code_map,
    load_config,
    resolve_path,
    setup_logging,
)
from src.voc_dataset import ImageXmlPair, VocObject, VocParseError, discover_dataset, parse_voc_xml, read_image_size

logger = logging.getLogger("convert_voc_to_yolo")

MANIFEST_NAME = "manifest.csv"
SOURCE_RDD = "rdd2022"
SOURCE_EXTRA = "extra"
EXTRA_UID_PREFIX = "extra__"
LABELS_SUBDIR = "labels"
YOLO_FLOAT_FORMAT = "{:.6f}"


@dataclass
class ConversionStats:
    """Counters accumulated over the whole conversion."""

    images_written: int = 0
    images_skipped: int = 0
    boxes_written: Counter = field(default_factory=Counter)
    ignored_labels: Counter = field(default_factory=Counter)
    boxes_clipped: int = 0
    boxes_swapped: int = 0
    boxes_dropped: int = 0
    issues: list[dict] = field(default_factory=list)

    def add_issue(self, file: Path, severity: str, message: str) -> None:
        self.issues.append({"file": str(file), "severity": severity, "message": message})


def to_yolo_box(
    obj: VocObject, width: int, height: int, min_size: float, stats: ConversionStats, xml_path: Path
) -> tuple[float, float, float, float] | None:
    """Validate/clip a VOC box and return normalised (cx, cy, w, h), or None if unusable."""
    x1, y1, x2, y2 = obj.xmin, obj.ymin, obj.xmax, obj.ymax
    if x1 > x2 or y1 > y2:
        x1, x2 = sorted((x1, x2))
        y1, y2 = sorted((y1, y2))
        stats.boxes_swapped += 1
        stats.add_issue(xml_path, "warning", f"{obj.name}: min/max coordinates swapped; fixed")

    cx1, cy1 = min(max(x1, 0.0), width), min(max(y1, 0.0), height)
    cx2, cy2 = min(max(x2, 0.0), width), min(max(y2, 0.0), height)
    if (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2):
        stats.boxes_clipped += 1
        stats.add_issue(
            xml_path, "warning",
            f"{obj.name}: box ({x1:g},{y1:g},{x2:g},{y2:g}) outside {width}x{height}; clipped",
        )

    bw, bh = cx2 - cx1, cy2 - cy1
    if bw < min_size or bh < min_size:
        stats.boxes_dropped += 1
        stats.add_issue(xml_path, "error", f"{obj.name}: degenerate box {bw:g}x{bh:g}px after clipping; dropped")
        return None
    return (cx1 + bw / 2) / width, (cy1 + bh / 2) / height, bw / width, bh / height


def image_dimensions(pair: ImageXmlPair, xml_w: int | None, xml_h: int | None, stats: ConversionStats):
    """Return the real image (width, height); report mismatches with the XML <size>."""
    try:
        width, height = read_image_size(pair.image_path)
    except OSError as exc:
        stats.add_issue(pair.image_path, "error", f"unreadable image ({exc}); skipped")
        return None
    if xml_w and xml_h and (xml_w, xml_h) != (width, height):
        stats.add_issue(pair.xml_path, "warning",
                        f"XML size {xml_w}x{xml_h} differs from image {width}x{height}; using image size")
    return width, height


def convert_pair(pair: ImageXmlPair, code_map: dict[str, int], labels_dir: Path,
                 min_size: float, stats: ConversionStats) -> dict | None:
    """Convert one image/XML pair; return a manifest row or None if skipped."""
    try:
        ann, problems = parse_voc_xml(pair.xml_path)
    except VocParseError as exc:
        stats.add_issue(pair.xml_path, "error", f"{exc}; image skipped")
        return None
    for msg in problems:
        stats.add_issue(pair.xml_path, "error", f"{msg}; object skipped")

    dims = image_dimensions(pair, ann.width, ann.height, stats)
    if dims is None:
        return None
    width, height = dims

    lines: list[str] = []
    counts: Counter = Counter()
    for obj in ann.objects:
        code = obj.name.strip().upper()
        if code not in code_map:
            stats.ignored_labels[code] += 1
            continue
        box = to_yolo_box(obj, width, height, min_size, stats, pair.xml_path)
        if box is None:
            continue
        class_id = code_map[code]
        counts[class_id] += 1
        lines.append(" ".join([str(class_id)] + [YOLO_FLOAT_FORMAT.format(v) for v in box]))

    (labels_dir / f"{pair.uid}.txt").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    stats.boxes_written.update(counts)
    row = {"uid": pair.uid, "image_path": pair.image_path.as_posix(), "xml_path": pair.xml_path.as_posix(),
           "width": width, "height": height, "num_boxes": len(lines)}
    row.update({f"n_class_{cid}": counts.get(cid, 0) for cid in sorted(set(code_map.values()))})
    return row


def collect_sources(raw_dir: Path, extra_dirs: list[Path], cfg: dict) -> list[tuple[str, Path, list]]:
    """Discover RDD2022 pairs plus pairs from each extra folder (images without XML are ignored)."""
    exts, ann_ext = get_image_extensions(cfg), cfg["dataset"]["annotation_extension"]
    rdd = discover_dataset(raw_dir, exts, ann_ext)
    inside_extra = lambda p: any(d in p.parents for d in extra_dirs)  # noqa: E731
    rdd_pairs = [p for p in rdd.pairs if not inside_extra(p.image_path)]
    if not rdd_pairs:
        raise FileNotFoundError(f"No RDD2022 image/XML pairs found under {raw_dir}. Run src/inspect_dataset.py first.")
    sources = [(SOURCE_RDD, raw_dir, rdd_pairs)]
    for extra in extra_dirs:
        if not extra.is_dir():
            raise FileNotFoundError(f"Extra dataset folder not found: {extra} (dataset.extra_dirs in config.yaml)")
        found = discover_dataset(extra, exts, ann_ext, uid_prefix=EXTRA_UID_PREFIX)
        logger.info("Extra folder %s: %d images, %d with XML (images without XML are not used).",
                    extra, len(found.images), len(found.pairs))
        sources.append((SOURCE_EXTRA, extra, found.pairs))
    return sources


def prepare_output_dir(interim_dir: Path, force: bool) -> Path:
    """Create (or, with --force, recreate) the interim labels directory."""
    labels_dir = interim_dir / LABELS_SUBDIR
    if labels_dir.exists() and any(labels_dir.iterdir()):
        if not force:
            raise FileExistsError(f"{labels_dir} already contains labels. Re-run with --force to overwrite.")
        shutil.rmtree(labels_dir)
    labels_dir.mkdir(parents=True, exist_ok=True)
    return labels_dir


def log_summary(stats: ConversionStats, names: dict[int, str], num_pairs: int) -> None:
    """Print the end-of-run summary."""
    logger.info("=" * 60)
    logger.info("Conversion summary")
    logger.info("  matched pairs          : %d", num_pairs)
    logger.info("  label files written    : %d", stats.images_written)
    logger.info("  images skipped         : %d", stats.images_skipped)
    for cid, name in names.items():
        logger.info("  boxes %-17s: %d", name, stats.boxes_written.get(cid, 0))
    logger.info("  boxes clipped to image : %d", stats.boxes_clipped)
    logger.info("  boxes with swapped mins: %d", stats.boxes_swapped)
    logger.info("  degenerate boxes dropped: %d", stats.boxes_dropped)
    ignored = sum(stats.ignored_labels.values())
    logger.info("  annotations ignored (non-target classes): %d %s", ignored, dict(stats.ignored_labels) or "")
    logger.info("=" * 60)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--raw-dir", default=None, help="Override dataset.raw_dir")
    parser.add_argument("--force", action="store_true", help="Overwrite existing converted labels")
    parser.add_argument("--no-extra", action="store_true", help="Ignore dataset.extra_dirs")
    args = parser.parse_args()

    setup_logging()
    cfg = load_config(args.config)
    raw_dir = resolve_path(args.raw_dir or cfg["dataset"]["raw_dir"])
    interim_dir = resolve_path(cfg["dataset"]["interim_dir"])
    reports_dir = resolve_path(cfg["paths"]["reports_dir"])
    code_map = get_source_code_map(cfg)
    min_size = float(cfg["dataset"].get("min_box_size_px", 1.0))

    extra_dirs = [] if args.no_extra else [resolve_path(d) for d in cfg["dataset"].get("extra_dirs") or []]
    try:
        sources = collect_sources(raw_dir, extra_dirs, cfg)
        labels_dir = prepare_output_dir(interim_dir, args.force)
    except (FileNotFoundError, FileExistsError) as exc:
        logger.error("%s", exc)
        return 1

    stats = ConversionStats()
    rows = []
    total_pairs = sum(len(pairs) for _, _, pairs in sources)
    for source, root, pairs in sources:
        logger.info("Converting %d image/XML pairs from %s (%s)", len(pairs), root, source)
        for pair in tqdm(pairs, desc=f"Converting {source}", unit="img"):
            row = convert_pair(pair, code_map, labels_dir, min_size, stats)
            if row is None:
                stats.images_skipped += 1
            else:
                stats.images_written += 1
                row["source"] = source
                rows.append(row)

    pd.DataFrame(rows).to_csv(interim_dir / MANIFEST_NAME, index=False)
    reports_dir.mkdir(parents=True, exist_ok=True)
    issues_path = reports_dir / "conversion_issues.csv"
    pd.DataFrame(stats.issues, columns=["file", "severity", "message"]).to_csv(issues_path, index=False)
    for issue in stats.issues[:10]:
        logger.warning("%s: %s", Path(issue["file"]).name, issue["message"])
    if len(stats.issues) > 10:
        logger.warning("... %d more issues, see %s", len(stats.issues) - 10, issues_path)

    log_summary(stats, get_class_names(cfg), total_pairs)
    if extra_dirs:
        per_source = pd.DataFrame(rows).groupby("source")[[c for c in rows[0] if c.startswith("n_class_")]].sum()
        per_source.columns = [get_class_names(cfg)[int(c.split("_")[-1])] for c in per_source.columns]
        logger.info("Boxes per source (extra = training only):\n%s", per_source.to_string())
    logger.info("Labels: %s", labels_dir)
    logger.info("Manifest: %s", interim_dir / MANIFEST_NAME)
    logger.info("Issues report: %s", issues_path)
    return 0 if rows else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/crop_defects.py
"""Save each road defect as a crop (+ metadata.csv) for severity labeling.

Two modes:

* detector (default) - run the trained detector on images and crop its detections.
* --ground-truth     - crop the annotated boxes of a prepared YOLO split. Needs no
                       trained detector, so labeling can start while training runs.

Usage:
    python src/crop_defects.py --source data/yolo/images/train --ground-truth --max-per-class 150 --save-crops
    python src/crop_defects.py --source data/yolo/images/test                   # detect only, print summary
    python src/crop_defects.py --source data/raw/RDD2022_India/India/test/images --save-crops
"""

from __future__ import annotations

import argparse
import logging
import random
import sys
from collections import defaultdict
from pathlib import Path
from typing import Any

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import cv2
import pandas as pd
from tqdm import tqdm

from src.common import (
    get_class_names,
    get_image_extensions,
    load_config,
    resolve_device,
    resolve_path,
    resolve_weights,
    setup_logging,
    to_project_relative,
)
from src.inference import crop_detection, load_detector, predict_image

logger = logging.getLogger("crop_defects")

METADATA_NAME = "metadata.csv"
METADATA_COLUMNS = ["image_name", "image_path", "image_width", "image_height", "crop_path", "class_id",
                    "class_name", "confidence", "x1", "y1", "x2", "y2", "source"]
SOURCE_DETECTOR = "detector"
SOURCE_GROUND_TRUTH = "ground_truth"


def list_images(source: Path, extensions: set[str]) -> list[Path]:
    """Return a single image or all images (recursively) under a directory."""
    if source.is_file():
        return [source]
    if source.is_dir():
        return sorted(p for p in source.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
    raise FileNotFoundError(f"Source not found: {source}")


def save_crops(image: Any, image_path: Path, detections: list[dict[str, Any]], crops_dir: Path,
               padding: float, min_size: int, source: str) -> list[dict[str, Any]]:
    """Save crops as <imageName>_<detection|gt>_<NNN>.jpg and return metadata rows.

    Each detection may carry an 'index' (used in the filename); otherwise its position is used.
    """
    tag = "gt" if source == SOURCE_GROUND_TRUTH else "detection"
    img_h, img_w = image.shape[:2]
    rows = []
    for pos, det in enumerate(detections, start=1):
        idx = det.get("index", pos)
        crop = crop_detection(image, det["bbox"], padding)
        if crop.shape[0] < min_size or crop.shape[1] < min_size:
            logger.debug("%s box %d: crop %dx%d below %dpx; not saved", image_path.name, idx,
                         crop.shape[1], crop.shape[0], min_size)
            continue
        crop_path = crops_dir / f"{image_path.stem}_{tag}_{idx:03d}.jpg"
        cv2.imwrite(str(crop_path), crop)
        x1, y1, x2, y2 = det["bbox"]
        rows.append({"image_name": image_path.name, "image_path": to_project_relative(image_path),
                     "image_width": img_w, "image_height": img_h, "crop_path": to_project_relative(crop_path),
                     "class_id": det["class_id"], "class_name": det["class_name"],
                     "confidence": det.get("confidence"), "x1": x1, "y1": y1, "x2": x2, "y2": y2, "source": source})
    return rows


def write_metadata(metadata_path: Path, new_rows: list[dict[str, Any]], replaced_images: set[str]) -> int:
    """Merge new rows into metadata.csv.

    Old rows are replaced when they have the same crop_path, or (detector mode)
    belong to an image that was re-processed.
    """
    new_df = pd.DataFrame(new_rows, columns=METADATA_COLUMNS)
    if metadata_path.is_file():
        old = pd.read_csv(metadata_path)
        stale = old["crop_path"].isin(new_df["crop_path"])
        if "source" in old:
            stale |= (old["source"] == SOURCE_DETECTOR) & old["image_name"].isin(replaced_images)
        new_df = pd.concat([old[~stale], new_df], ignore_index=True)
    new_df.to_csv(metadata_path, index=False)
    return len(new_df)


def labels_dir_for(images_dir: Path) -> Path:
    """data/yolo/images/<split> -> data/yolo/labels/<split>."""
    return images_dir.parent.parent / "labels" / images_dir.name


def read_ground_truth(images: list[Path], labels_dir: Path) -> list[dict[str, Any]]:
    """Read YOLO label files; returns one record per annotated box (normalised coordinates)."""
    records = []
    for image_path in images:
        label_path = labels_dir / f"{image_path.stem}.txt"
        if not label_path.is_file():
            continue
        for idx, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
            parts = line.split()
            if len(parts) != 5:
                continue
            class_id, (cx, cy, w, h) = int(parts[0]), map(float, parts[1:])
            records.append({"image_path": image_path, "index": idx, "class_id": class_id, "box": (cx, cy, w, h)})
    return records


def sample_per_class(records: list[dict[str, Any]], max_per_class: int | None, seed: int) -> list[dict[str, Any]]:
    """Randomly keep at most ``max_per_class`` boxes of each class (reproducible)."""
    if not max_per_class:
        return records
    by_class: dict[int, list] = defaultdict(list)
    for rec in records:
        by_class[rec["class_id"]].append(rec)
    rng = random.Random(seed)
    kept = []
    for class_id in sorted(by_class):
        items = by_class[class_id]
        kept.extend(rng.sample(items, min(max_per_class, len(items))))
    return kept


def to_pixel_bbox(box: tuple[float, float, float, float], width: int, height: int) -> list[int]:
    cx, cy, w, h = box
    return [int(round((cx - w / 2) * width)), int(round((cy - h / 2) * height)),
            int(round((cx + w / 2) * width)), int(round((cy + h / 2) * height))]


def run_ground_truth(args: argparse.Namespace, cfg: dict[str, Any], crops_dir: Path) -> tuple[list, set, int]:
    """Crop annotated boxes from a YOLO split."""
    source = resolve_path(args.source)
    labels_dir = labels_dir_for(source)
    if not labels_dir.is_dir():
        raise FileNotFoundError(f"--ground-truth needs YOLO labels at {labels_dir} "
                                "(use --source data/yolo/images/<split>).")
    names = get_class_names(cfg)
    records = read_ground_truth(list_images(source, get_image_extensions(cfg)), labels_dir)
    records = sample_per_class(records, args.max_per_class, int(cfg["split"]["seed"]))
    counts = pd.Series([names[r["class_id"]] for r in records]).value_counts().to_dict() if records else {}
    logger.info("Ground-truth boxes selected: %d %s", len(records), counts)

    by_image: dict[Path, list] = defaultdict(list)
    for rec in records:
        by_image[rec["image_path"]].append(rec)
    rows: list[dict[str, Any]] = []
    for image_path, recs in tqdm(sorted(by_image.items()), desc="Cropping", unit="img"):
        image = cv2.imread(str(image_path))
        if image is None:
            logger.warning("Could not read %s; skipped", image_path)
            continue
        h, w = image.shape[:2]
        dets = [{"index": r["index"], "class_id": r["class_id"], "class_name": names[r["class_id"]],
                 "bbox": to_pixel_bbox(r["box"], w, h)} for r in sorted(recs, key=lambda r: r["index"])]
        if args.save_crops:
            rows.extend(save_crops(image, image_path, dets, crops_dir, float(cfg["severity"]["crop_padding"]),
                                   int(cfg["severity"]["min_crop_size_px"]), SOURCE_GROUND_TRUTH))
    return rows, set(), len(records)


def run_detector(args: argparse.Namespace, cfg: dict[str, Any], crops_dir: Path) -> tuple[list, set, int]:
    """Run the detector and crop its detections."""
    weights = resolve_weights(cfg, args.weights)
    images = list_images(resolve_path(args.source), get_image_extensions(cfg))
    if not images:
        raise FileNotFoundError(f"No images found in {args.source}")
    model = load_detector(weights)
    device = resolve_device(args.device or cfg["training"].get("device", "auto"))
    conf = args.conf if args.conf is not None else cfg["model"]["confidence"]
    iou = args.iou if args.iou is not None else cfg["model"]["iou"]

    rows: list[dict[str, Any]] = []
    processed: set[str] = set()
    total = 0
    for path in tqdm(images, desc="Detecting", unit="img"):
        image = cv2.imread(str(path))
        if image is None:
            logger.warning("Could not read %s; skipped", path)
            continue
        detections = predict_image(model, image, conf=conf, iou=iou, device=device)
        total += len(detections)
        processed.add(path.name)
        if args.save_crops:
            rows.extend(save_crops(image, path, detections, crops_dir, float(cfg["severity"]["crop_padding"]),
                                   int(cfg["severity"]["min_crop_size_px"]), SOURCE_DETECTOR))
    logger.info("Processed %d images, %d detections.", len(processed), total)
    return rows, processed, total


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--source", required=True, help="Image file or directory of images")
    parser.add_argument("--ground-truth", action="store_true",
                        help="Crop annotated boxes of a YOLO split (--source data/yolo/images/<split>)")
    parser.add_argument("--max-per-class", type=int, default=None,
                        help="Ground-truth mode: randomly keep at most N boxes per class")
    parser.add_argument("--weights", default=None, help="Detector weights (default: inference.weights)")
    parser.add_argument("--save-crops", action="store_true", help="Save crops + metadata.csv to severity.crops_dir")
    parser.add_argument("--output-dir", default=None, help="Override severity.crops_dir")
    parser.add_argument("--conf", type=float, default=None, help="Confidence threshold")
    parser.add_argument("--iou", type=float, default=None, help="NMS IoU threshold")
    parser.add_argument("--device", default=None)
    args = parser.parse_args()

    setup_logging()
    cfg = load_config(args.config)
    crops_dir = resolve_path(args.output_dir or cfg["severity"]["crops_dir"])
    if args.save_crops:
        crops_dir.mkdir(parents=True, exist_ok=True)
    try:
        runner = run_ground_truth if args.ground_truth else run_detector
        rows, processed, _ = runner(args, cfg, crops_dir)
    except FileNotFoundError as exc:
        logger.error("%s", exc)
        return 1

    if args.save_crops:
        n = write_metadata(crops_dir / METADATA_NAME, rows, processed)
        logger.info("Saved %d crops to %s (metadata.csv now has %d rows).", len(rows), crops_dir, n)
        logger.info("Next: label them with `streamlit run label_severity.py`.")
    else:
        logger.info("Crops not saved (use --save-crops to save them).")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/evaluate_detector.py
"""Evaluate the trained detector on the held-out test split.

Reports precision, recall, mAP50 and mAP50-95 (overall and per class), and
saves Ultralytics plots (confusion matrix, PR curves) plus example predictions
under ``reports/evaluation/``.

Usage:
    python src/evaluate_detector.py [--weights models/road_damage_yolo.pt] [--split test]
"""

from __future__ import annotations

import argparse
import json
import logging
import random
import sys
from pathlib import Path
from typing import Any

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import pandas as pd

from src.common import get_image_extensions, load_config, resolve_device, resolve_path, resolve_weights, setup_logging

logger = logging.getLogger("evaluate_detector")

EVAL_SUBDIR = "evaluation"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--weights", default=None, help="Detector weights (default: inference.weights / latest run)")
    parser.add_argument("--split", default=None, choices=["train", "val", "test"])
    parser.add_argument("--device", default=None)
    parser.add_argument("--imgsz", type=int, default=None)
    parser.add_argument("--batch", type=int, default=None)
    return parser.parse_args()


def metrics_tables(metrics: Any, names: dict[int, str]) -> tuple[dict[str, float], pd.DataFrame]:
    """Extract overall and per-class metrics from an Ultralytics DetMetrics object."""
    box = metrics.box
    overall = {"precision": float(box.mp), "recall": float(box.mr),
               "mAP50": float(box.map50), "mAP50-95": float(box.map)}
    rows = []
    for i, class_id in enumerate(box.ap_class_index):
        p, r, ap50, ap = box.class_result(i)
        rows.append({"class_id": int(class_id), "class_name": names[int(class_id)], "precision": float(p),
                     "recall": float(r), "mAP50": float(ap50), "mAP50-95": float(ap)})
    return overall, pd.DataFrame(rows)


def save_example_predictions(model: Any, image_dir: Path, out_dir: Path, count: int, cfg: dict[str, Any],
                             device: str, seed: int) -> int:
    """Run the detector on a few images and save annotated copies."""
    import cv2

    from src.inference import draw_predictions, predict_image

    images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in get_image_extensions(cfg))
    if not images:
        return 0
    random.Random(seed).shuffle(images)
    out_dir.mkdir(parents=True, exist_ok=True)
    for path in images[:count]:
        image = cv2.imread(str(path))
        if image is None:
            logger.warning("Could not read %s", path)
            continue
        detections = predict_image(model, image, conf=cfg["model"]["confidence"], iou=cfg["model"]["iou"],
                                   device=device)
        annotated = draw_predictions(image, detections, cfg["inference"]["box_thickness"],
                                     cfg["inference"]["font_scale"])
        cv2.imwrite(str(out_dir / f"{path.stem}_pred.jpg"), annotated)
    return min(count, len(images))


def main() -> int:
    args = parse_args()
    setup_logging()
    cfg = load_config(args.config)
    ev = cfg["evaluation"]
    split = args.split or ev.get("split", "test")
    data_yaml = resolve_path(cfg["dataset"]["yolo_dir"]) / "data.yaml"
    reports_dir = resolve_path(cfg["paths"]["reports_dir"])
    device = resolve_device(args.device or cfg["training"].get("device", "auto"))

    try:
        weights = resolve_weights(cfg, args.weights)
        if not data_yaml.is_file():
            raise FileNotFoundError(f"{data_yaml} not found. Run src/split_dataset.py first.")
        split_dir = data_yaml.parent / "images" / split
        if not split_dir.is_dir() or not any(split_dir.iterdir()):
            raise FileNotFoundError(f"No images in {split_dir}; cannot evaluate split '{split}'.")
    except FileNotFoundError as exc:
        logger.error("%s", exc)
        return 1

    from src.inference import load_detector

    model = load_detector(weights)
    names = {int(k): v for k, v in model.names.items()}
    logger.info("Evaluating %s on split '%s' (device=%s)", weights, split, device)
    metrics = model.val(
        data=data_yaml.as_posix(), split=split, device=device,
        imgsz=args.imgsz or cfg["training"]["image_size"], batch=args.batch or cfg["training"]["batch"],
        conf=ev["confidence"], iou=ev["iou"], plots=True,
        project=reports_dir.as_posix(), name=EVAL_SUBDIR, exist_ok=True,
    )
    eval_dir = Path(getattr(metrics, "save_dir", reports_dir / EVAL_SUBDIR))
    overall, per_class = metrics_tables(metrics, names)

    logger.info("=" * 60)
    logger.info("Overall (%s): P=%.3f  R=%.3f  mAP50=%.3f  mAP50-95=%.3f", split, overall["precision"],
                overall["recall"], overall["mAP50"], overall["mAP50-95"])
    if not per_class.empty:
        logger.info("Per class:\n%s", per_class.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    logger.info("=" * 60)

    eval_dir.mkdir(parents=True, exist_ok=True)
    per_class.to_csv(eval_dir / "per_class_metrics.csv", index=False)
    (eval_dir / "metrics.json").write_text(json.dumps(
        {"weights": weights.as_posix(), "split": split, "overall": overall,
         "per_class": per_class.to_dict(orient="records")}, indent=2), encoding="utf-8")

    n_examples = save_example_predictions(model, split_dir, eval_dir / "examples", int(ev["num_example_images"]),
                                          cfg, device, int(cfg["split"]["seed"]))
    plots = sorted(p.name for p in eval_dir.glob("*.png"))
    logger.info("Saved metrics.json, per_class_metrics.csv, %d example predictions and plots %s in %s",
                n_examples, plots, eval_dir)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/inference.py
"""Reusable detector inference: loading, prediction, drawing and severity hookup.

Used by the Streamlit app, evaluate_detector.py and crop_defects.py so that no
inference logic lives in the UI.
"""

from __future__ import annotations

import logging
from pathlib import Path
from typing import TYPE_CHECKING, Any

import cv2
import numpy as np
from PIL import Image

if TYPE_CHECKING:
    from ultralytics import YOLO

    from src.severity import HeuristicSeverityEstimator, SeverityPredictor

    SeverityEstimator = SeverityPredictor | HeuristicSeverityEstimator

logger = logging.getLogger(__name__)

DEFAULT_BOX_THICKNESS = 2
DEFAULT_FONT_SCALE = 0.6
LABEL_PADDING_PX = 4
# Fixed per-class colours (BGR), in the project palette order: blue, orange, aqua, yellow.
CLASS_COLORS_BGR: dict[int, tuple[int, int, int]] = {
    0: (214, 120, 42),
    1: (52, 104, 235),
    2: (122, 175, 27),
    3: (0, 161, 237),
}
FALLBACK_COLOR_BGR = (129, 135, 137)
DARK_TEXT_BGR = (11, 11, 11)
LIGHT_TEXT_BGR = (255, 255, 255)
LUMINANCE_TEXT_THRESHOLD = 150


def load_detector(weights_path: str | Path) -> "YOLO":
    """Load an Ultralytics YOLO model from a weights file."""
    path = Path(weights_path)
    if not path.is_file():
        raise FileNotFoundError(f"Detector weights not found: {path}")
    try:
        from ultralytics import YOLO
    except ImportError as exc:
        raise ImportError("Ultralytics is not installed. Run: pip install -r requirements.txt") from exc
    return YOLO(str(path))


def to_bgr_array(image: np.ndarray | Image.Image | str | Path) -> np.ndarray:
    """Convert a path, PIL image (RGB) or BGR array into a BGR uint8 array."""
    if isinstance(image, np.ndarray):
        return image
    if isinstance(image, (str, Path)):
        array = cv2.imread(str(image))
        if array is None:
            raise ValueError(f"Could not read image: {image}")
        return array
    if isinstance(image, Image.Image):
        return cv2.cvtColor(np.asarray(image.convert("RGB")), cv2.COLOR_RGB2BGR)
    raise TypeError(f"Unsupported image type: {type(image)!r}")


def predict_image(
    model: "YOLO",
    image: np.ndarray | Image.Image | str | Path,
    conf: float = 0.25,
    iou: float = 0.45,
    device: str | None = None,
    imgsz: int | None = None,
) -> list[dict[str, Any]]:
    """Run the detector on one image.

    Returns a list of dicts sorted by confidence::

        {"class_id": 3, "class_name": "pothole", "confidence": 0.94, "bbox": [x1, y1, x2, y2]}

    Bounding boxes are integer pixel coordinates in the input image.
    """
    kwargs: dict[str, Any] = {"conf": conf, "iou": iou, "verbose": False}
    if device is not None:
        kwargs["device"] = device
    if imgsz is not None:
        kwargs["imgsz"] = imgsz
    result = model.predict(to_bgr_array(image), **kwargs)[0]

    detections = []
    for xyxy, score, cls in zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(), result.boxes.cls.tolist()):
        class_id = int(cls)
        detections.append({
            "class_id": class_id,
            "class_name": str(result.names.get(class_id, class_id)),
            "confidence": round(float(score), 4),
            "bbox": [int(round(v)) for v in xyxy],
        })
    return sorted(detections, key=lambda d: d["confidence"], reverse=True)


def _text_color(bg: tuple[int, int, int]) -> tuple[int, int, int]:
    b, g, r = bg
    luminance = 0.299 * r + 0.587 * g + 0.114 * b
    return DARK_TEXT_BGR if luminance > LUMINANCE_TEXT_THRESHOLD else LIGHT_TEXT_BGR


def format_label(det: dict[str, Any], show_confidence: bool = True) -> str:
    """Label text drawn above a box, including severity only when available."""
    label = f"{det['class_name']} {det['confidence']:.2f}" if show_confidence else det["class_name"]
    if det.get("severity"):
        label += f" | {det['severity']}"
        if det.get("severity_method") == "heuristic":
            label += " (est.)"
    return label


def draw_predictions(
    image: np.ndarray,
    detections: list[dict[str, Any]],
    box_thickness: int = DEFAULT_BOX_THICKNESS,
    font_scale: float = DEFAULT_FONT_SCALE,
    show_confidence: bool = True,
) -> np.ndarray:
    """Return a copy of a BGR image with boxes and 'class confidence' labels drawn.

    Use ``show_confidence=False`` for human annotations, which have no confidence.
    """
    canvas = image.copy()
    font = cv2.FONT_HERSHEY_SIMPLEX
    text_thickness = max(1, box_thickness - 1)
    for det in detections:
        x1, y1, x2, y2 = det["bbox"]
        color = CLASS_COLORS_BGR.get(det["class_id"], FALLBACK_COLOR_BGR)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, box_thickness)

        label = format_label(det, show_confidence)
        (tw, th), baseline = cv2.getTextSize(label, font, font_scale, text_thickness)
        top = y1 - th - baseline - LABEL_PADDING_PX
        if top < 0:  # no room above the box: draw the label inside it
            top = y1
        x1 = max(0, min(x1, canvas.shape[1] - tw - LABEL_PADDING_PX))  # keep label inside the image
        cv2.rectangle(canvas, (x1, top), (x1 + tw + LABEL_PADDING_PX, top + th + baseline + LABEL_PADDING_PX),
                      color, cv2.FILLED)
        cv2.putText(canvas, label, (x1 + LABEL_PADDING_PX // 2, top + th + LABEL_PADDING_PX // 2),
                    font, font_scale, _text_color(color), text_thickness, cv2.LINE_AA)
    return canvas


def crop_detection(image: np.ndarray, bbox: list[int], padding: float = 0.0) -> np.ndarray:
    """Crop a box from a BGR image, expanded by ``padding`` (fraction of box size) and clipped."""
    h, w = image.shape[:2]
    x1, y1, x2, y2 = bbox
    pad_x, pad_y = (x2 - x1) * padding, (y2 - y1) * padding
    left, top = max(0, int(x1 - pad_x)), max(0, int(y1 - pad_y))
    right, bottom = min(w, int(round(x2 + pad_x))), min(h, int(round(y2 + pad_y)))
    return image[top:bottom, left:right].copy()


def attach_severity(
    image: np.ndarray,
    detections: list[dict[str, Any]],
    severity_estimator: "SeverityEstimator | None",
    padding: float = 0.0,
) -> list[dict[str, Any]]:
    """Add 'severity', 'severity_confidence' and 'severity_method' to each detection.

    'severity_method' is "model" (trained classifier), "heuristic" (rule-based
    estimate) or None (no severity available).
    """
    for det in detections:
        result = {"severity": None, "confidence": None, "method": None}
        if severity_estimator is not None and severity_estimator.available:
            crop = crop_detection(image, det["bbox"], padding)
            result = severity_estimator.predict(crop, detection=det, image_shape=image.shape[:2])
        det["severity"] = result["severity"]
        det["severity_confidence"] = result["confidence"]
        det["severity_method"] = result["method"]
    return detections


def analyze_image(
    model: "YOLO",
    image: np.ndarray | Image.Image | str | Path,
    severity_estimator: "SeverityEstimator | None" = None,
    conf: float = 0.25,
    iou: float = 0.45,
    device: str | None = None,
    crop_padding: float = 0.0,
) -> tuple[np.ndarray, list[dict[str, Any]]]:
    """Full pipeline: detect defects, then (optionally) estimate severity per crop.

    Returns the BGR image and the detections with severity fields attached.
    """
    bgr = to_bgr_array(image)
    detections = predict_image(model, bgr, conf=conf, iou=iou, device=device)
    return bgr, attach_severity(bgr, detections, severity_estimator, crop_padding)


In [ ]:
%%writefile /content/proj/src/inspect_dataset.py
"""Inspect the raw RDD2022 download: file discovery, matching and class distribution.

Usage:
    python src/inspect_dataset.py [--config config.yaml] [--raw-dir PATH]
"""

from __future__ import annotations

import argparse
import logging
import sys
from collections import Counter
from pathlib import Path

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from src.common import (
    get_class_names,
    get_image_extensions,
    get_source_code_map,
    load_config,
    resolve_path,
    setup_logging,
)
from src.voc_dataset import DiscoveryResult, VocParseError, discover_dataset, parse_voc_xml, read_image_size

logger = logging.getLogger("inspect_dataset")

MAX_LISTED_FILES = 10
# Colours from the project's chart palette: target classes vs. ignored labels.
TARGET_BAR_COLOR = "#2a78d6"
IGNORED_BAR_COLOR = "#898781"
TEXT_COLOR = "#52514e"


def summarize_directories(result: DiscoveryResult, root: Path) -> None:
    """Log which folders contain images and which contain XML annotations."""
    for label, files in (("Image", result.images), ("Annotation (XML)", result.xmls)):
        counts = Counter(str(p.parent.relative_to(root)) for p in files)
        logger.info("%s directories:", label)
        for folder, n in counts.most_common():
            logger.info("    %-60s %6d files", folder, n)


def log_file_list(title: str, files: list[Path], root: Path) -> None:
    """Log up to MAX_LISTED_FILES entries of a file list."""
    if not files:
        return
    logger.warning("%s (%d), first %d:", title, len(files), min(len(files), MAX_LISTED_FILES))
    for p in files[:MAX_LISTED_FILES]:
        logger.warning("    %s", p.relative_to(root))


def collect_annotation_stats(result: DiscoveryResult) -> tuple[Counter, list[tuple[int, int]], list[str]]:
    """Parse all XML files; return label counts, image sizes and problem descriptions."""
    label_counts: Counter = Counter()
    sizes: list[tuple[int, int]] = []
    problems: list[str] = []
    image_for_xml = {p.xml_path: p.image_path for p in result.pairs}

    for xml_path in result.xmls:
        try:
            ann, obj_problems = parse_voc_xml(xml_path)
        except VocParseError as exc:
            problems.append(f"{xml_path}: {exc}")
            continue
        problems.extend(f"{xml_path}: {msg}" for msg in obj_problems)
        label_counts.update(o.name.strip().upper() for o in ann.objects)

        image_path = image_for_xml.get(xml_path)
        if image_path is None:
            continue
        try:
            sizes.append(read_image_size(image_path))
        except OSError as exc:
            problems.append(f"{image_path}: unreadable image ({exc})")
    return label_counts, sizes, problems


def build_distribution_table(label_counts: Counter, cfg: dict) -> pd.DataFrame:
    """Create a class distribution table including ignored (non-target) labels."""
    names = get_class_names(cfg)
    code_map = get_source_code_map(cfg)
    rows = []
    for code, class_id in sorted(code_map.items(), key=lambda kv: kv[1]):
        rows.append({"code": code, "class_id": class_id, "class_name": names[class_id],
                     "instances": label_counts.get(code, 0), "used": True})
    for code, n in sorted(label_counts.items()):
        if code not in code_map:
            rows.append({"code": code, "class_id": None, "class_name": "(ignored)", "instances": n, "used": False})
    df = pd.DataFrame(rows)
    df["class_id"] = df["class_id"].astype("Int64")
    total = df["instances"].sum()
    df["percent"] = (100.0 * df["instances"] / total).round(2) if total else 0.0
    return df


def plot_distribution(df: pd.DataFrame, out_path: Path) -> None:
    """Save a bar chart of instances per label code."""
    fig, ax = plt.subplots(figsize=(max(6, 1.4 * len(df)), 4.5))
    colors = [TARGET_BAR_COLOR if used else IGNORED_BAR_COLOR for used in df["used"]]
    labels = [f"{c}\n{n.replace('_', chr(10))}" if u else f"{c}\n(ignored)"
              for c, n, u in zip(df["code"], df["class_name"], df["used"])]
    bars = ax.bar(labels, df["instances"], color=colors, width=0.6)
    ax.bar_label(bars, labels=[f"{v:,}" for v in df["instances"]], padding=3, fontsize=9, color=TEXT_COLOR)
    ax.set_ylabel("Instances", color=TEXT_COLOR)
    ax.set_title("RDD2022 India - annotated instances per class", loc="left")
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=TEXT_COLOR, labelsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def log_size_stats(sizes: list[tuple[int, int]]) -> None:
    """Log image width/height statistics."""
    if not sizes:
        logger.info("No image dimensions collected.")
        return
    df = pd.DataFrame(sizes, columns=["width", "height"])
    logger.info("Image dimensions (%d images):", len(df))
    logger.info("    width : min %d  max %d  mean %.1f", df.width.min(), df.width.max(), df.width.mean())
    logger.info("    height: min %d  max %d  mean %.1f", df.height.min(), df.height.max(), df.height.mean())
    for (w, h), n in Counter(sizes).most_common(5):
        logger.info("    %dx%d: %d images", w, h, n)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--raw-dir", default=None, help="Override dataset.raw_dir")
    args = parser.parse_args()

    setup_logging()
    cfg = load_config(args.config)
    root = resolve_path(args.raw_dir or cfg["dataset"]["raw_dir"])
    reports_dir = resolve_path(cfg["paths"]["reports_dir"])

    try:
        result = discover_dataset(root, get_image_extensions(cfg), cfg["dataset"]["annotation_extension"])
    except FileNotFoundError as exc:
        logger.error("%s", exc)
        return 1

    logger.info("Scanned: %s", root)
    logger.info("Images found              : %d", len(result.images))
    logger.info("XML files found           : %d", len(result.xmls))
    logger.info("Matched image/XML pairs   : %d", len(result.pairs))
    logger.info("Unmatched images          : %d", len(result.unmatched_images))
    logger.info("Unmatched XML files       : %d", len(result.unmatched_xmls))
    if not result.images and not result.xmls:
        logger.error("No images or XML files found. Check that the download is extracted into %s", root)
        return 1
    summarize_directories(result, root)
    log_file_list("Images without XML (not used for training; e.g. the unlabeled test split)",
                  result.unmatched_images, root)
    log_file_list("XML files without an image", result.unmatched_xmls, root)

    label_counts, sizes, problems = collect_annotation_stats(result)
    df = build_distribution_table(label_counts, cfg)
    logger.info("Class distribution (all XML files):\n%s", df.to_string(index=False))
    log_size_stats(sizes)

    if problems:
        logger.warning("%d annotation problems found, first %d:", len(problems), MAX_LISTED_FILES)
        for msg in problems[:MAX_LISTED_FILES]:
            logger.warning("    %s", msg)

    reports_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(reports_dir / "class_distribution.csv", index=False)
    plot_distribution(df, reports_dir / "class_distribution.png")
    if problems:
        (reports_dir / "inspection_problems.txt").write_text("\n".join(problems), encoding="utf-8")
    logger.info("Saved %s and %s", reports_dir / "class_distribution.csv", reports_dir / "class_distribution.png")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/severity.py
"""Severity estimation (Low / Medium / High) for detected road defects.

Why severity needs special care
-------------------------------
RDD2022 provides damage *type* labels (D00, D10, D20, D40, ...) and bounding
boxes only. It contains NO Low/Medium/High severity ground truth. Two
estimators are therefore provided, selected by ``severity.method`` in config.yaml:

1. ``SeverityPredictor`` - a trained image classifier (MobileNetV3 / ResNet18).
   This is the proper solution. Its training data comes from our own labeling:
     a. ``python src/crop_defects.py --source data/yolo/images/train --ground-truth --save-crops``
     b. ``streamlit run label_severity.py`` (rules in docs/severity_rubric.md)
     c. ``python src/train_severity.py`` -> models/severity_classifier.pt

2. ``HeuristicSeverityEstimator`` - a transparent rule-based ESTIMATE used until
   a trained classifier exists. It combines two measurable image cues:
     * size   - how large the defect is relative to the image (crack length for
                linear cracks, area for potholes / alligator cracking);
     * density - fraction of dark pixels inside the box (a proxy for crack
                width / spalling or pothole depth / shadow).
   The thresholds are documented starting values in config.yaml, NOT learned
   from ground truth. Its output is always marked ``method="heuristic"`` and the
   UI labels it as an estimate. train_severity.py reports how well it agrees
   with human labels once they exist.

``create_severity_estimator(cfg)`` returns the right estimator: with
``method: auto`` the trained model is used when its file exists, otherwise the
heuristic.

Checkpoint format for the trained model (a ``torch.save``'d dict)::

    {
        "arch": "mobilenet_v3_small" | "mobilenet_v3_large" | "resnet18",
        "state_dict": model.state_dict(),
        "labels": ["Low", "Medium", "High"],   # index order of the output logits
        "image_size": 224,                     # optional, default 224
        "mean": [0.485, 0.456, 0.406],         # optional, ImageNet defaults
        "std": [0.229, 0.224, 0.225],          # optional
    }
"""

from __future__ import annotations

import logging
from pathlib import Path
from typing import Any

import numpy as np

logger = logging.getLogger(__name__)

SEVERITY_LEVELS = ("Low", "Medium", "High")
SUPPORTED_ARCHS = ("mobilenet_v3_small", "mobilenet_v3_large", "resnet18")
DEFAULT_IMAGE_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
DENSITY_BLUR_KERNEL = (5, 5)
METHOD_MODEL = "model"
METHOD_HEURISTIC = "heuristic"


def not_available() -> dict[str, Any]:
    """Result used when no severity estimate can be made."""
    return {"severity": None, "confidence": None, "method": None}


def build_classifier(arch: str, num_classes: int, pretrained: bool = False):
    """Create a torchvision classifier with ``num_classes`` outputs (optionally ImageNet-pretrained)."""
    import torch.nn as nn
    from torchvision import models

    if arch not in SUPPORTED_ARCHS:
        raise ValueError(f"Unsupported severity architecture '{arch}'. Use one of {SUPPORTED_ARCHS}.")
    model = getattr(models, arch)(weights="DEFAULT" if pretrained else None)
    if arch == "resnet18":
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    return model


def preprocess_crop(crop_bgr: np.ndarray, image_size: int, mean=IMAGENET_MEAN, std=IMAGENET_STD) -> np.ndarray:
    """BGR crop -> normalised float32 CHW array (shared by training and inference)."""
    import cv2

    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
    array = (resized.astype(np.float32) / 255.0 - np.array(mean, np.float32)) / np.array(std, np.float32)
    return array.transpose(2, 0, 1)


def torch_device(device: str | None) -> str:
    """'cpu' / '0' / 'cuda:0' -> a torch device string."""
    if device in (None, "cpu"):
        return "cpu"
    return f"cuda:{device}" if str(device).isdigit() else str(device)


class SeverityPredictor:
    """Trained severity classifier for a single defect crop.

    If no model path is given, or the file does not exist, the predictor is
    "unavailable" and ``predict`` returns ``severity=None``.
    """

    method = METHOD_MODEL

    def __init__(self, model_path: str | Path | None = None, device: str = "cpu") -> None:
        self.model_path = Path(model_path) if model_path else None
        self.device = torch_device(device)
        self.model = None
        self.labels: list[str] = list(SEVERITY_LEVELS)
        self.image_size = DEFAULT_IMAGE_SIZE
        self.mean, self.std = IMAGENET_MEAN, IMAGENET_STD
        if self.model_path is None:
            return
        if not self.model_path.is_file():
            logger.warning("Severity model not found at %s; model-based severity disabled.", self.model_path)
            return
        self._load()

    @property
    def available(self) -> bool:
        """True when a trained severity model is loaded."""
        return self.model is not None

    def _load(self) -> None:
        import torch

        checkpoint = torch.load(self.model_path, map_location=self.device, weights_only=False)
        if not isinstance(checkpoint, dict) or "state_dict" not in checkpoint or "arch" not in checkpoint:
            raise ValueError(f"{self.model_path} is not a severity checkpoint (see src/severity.py docstring).")
        self.labels = list(checkpoint.get("labels", SEVERITY_LEVELS))
        self.image_size = int(checkpoint.get("image_size", DEFAULT_IMAGE_SIZE))
        self.mean = tuple(checkpoint.get("mean", IMAGENET_MEAN))
        self.std = tuple(checkpoint.get("std", IMAGENET_STD))
        model = build_classifier(checkpoint["arch"], len(self.labels))
        model.load_state_dict(checkpoint["state_dict"])
        self.model = model.to(self.device).eval()
        logger.info("Loaded severity model %s (%s, labels=%s)", self.model_path, checkpoint["arch"], self.labels)

    def predict(self, crop: np.ndarray, detection: dict | None = None,
                image_shape: tuple[int, int] | None = None) -> dict[str, Any]:
        """Classify one BGR crop. ``detection``/``image_shape`` are accepted for interface parity."""
        if not self.available or crop is None or crop.size == 0:
            return not_available()
        import torch

        tensor = torch.from_numpy(preprocess_crop(crop, self.image_size, self.mean, self.std)).unsqueeze(0)
        with torch.no_grad():
            probs = torch.softmax(self.model(tensor.to(self.device)), dim=1)[0]
        idx = int(probs.argmax())
        return {"severity": self.labels[idx], "confidence": round(float(probs[idx]), 4), "method": self.method}


def dark_pixel_fraction(crop_bgr: np.ndarray, dark_delta: float) -> float:
    """Fraction of pixels noticeably darker than the crop's median grey level."""
    import cv2

    gray = cv2.GaussianBlur(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY), DENSITY_BLUR_KERNEL, 0)
    return float((gray < np.median(gray) - dark_delta).mean())


class HeuristicSeverityEstimator:
    """Transparent rule-based severity ESTIMATE (see module docstring).

    score = size_weight * size_score + (1 - size_weight) * density_score, both in [0, 1];
    score < thresholds[0] -> Low, < thresholds[1] -> Medium, otherwise High.
    """

    method = METHOD_HEURISTIC
    available = True

    def __init__(self, params: dict[str, Any]) -> None:
        self.dark_delta = float(params["dark_delta"])
        self.size_ref = {k: float(v) for k, v in params["size_ref"].items()}
        self.density_ref = float(params["density_ref"])
        self.size_weight = float(params["size_weight"])
        self.low_max, self.medium_max = (float(t) for t in params["thresholds"])
        self.linear_classes = set(params.get("linear_classes", []))

    def features(self, crop: np.ndarray, detection: dict, image_shape: tuple[int, int]) -> dict[str, float]:
        """Size and density cues, each normalised to [0, 1]."""
        img_h, img_w = image_shape[:2]
        x1, y1, x2, y2 = detection["bbox"]
        box_w, box_h = max(0, x2 - x1), max(0, y2 - y1)
        if detection.get("class_name") in self.linear_classes:
            size = max(box_w / img_w, box_h / img_h)  # crack length along its main direction
            ref = self.size_ref["linear"]
        else:
            size = float(np.sqrt(box_w * box_h / (img_w * img_h)))  # linear size of an area defect
            ref = self.size_ref["area"]
        density = dark_pixel_fraction(crop, self.dark_delta)
        return {"size": min(size / ref, 1.0), "density": min(density / self.density_ref, 1.0)}

    def predict(self, crop: np.ndarray, detection: dict | None = None,
                image_shape: tuple[int, int] | None = None) -> dict[str, Any]:
        """Estimate severity; needs the detection (class_name, bbox) and the full image shape."""
        if crop is None or crop.size == 0 or detection is None or image_shape is None:
            return not_available()
        f = self.features(crop, detection, image_shape)
        score = self.size_weight * f["size"] + (1.0 - self.size_weight) * f["density"]
        level = "Low" if score < self.low_max else "Medium" if score < self.medium_max else "High"
        return {"severity": level, "confidence": None, "method": self.method, "score": round(score, 3)}


def create_severity_estimator(cfg: dict[str, Any], device: str = "cpu"):
    """Return the estimator selected by ``severity.method``: auto | model | heuristic | none."""
    from src.common import resolve_path

    sev = cfg["severity"]
    method = str(sev.get("method", "auto")).lower()
    model_path = resolve_path(sev.get("model_path"))
    if method in ("auto", METHOD_MODEL):
        if model_path and model_path.is_file():
            return SeverityPredictor(model_path, device=device)
        if method == METHOD_MODEL:
            logger.warning("severity.method is 'model' but %s does not exist; severity disabled.", model_path)
            return SeverityPredictor(None)
    if method in ("auto", METHOD_HEURISTIC):
        return HeuristicSeverityEstimator(sev["heuristic"])
    return SeverityPredictor(None)


In [ ]:
%%writefile /content/proj/src/split_dataset.py
"""Split converted data into train/val/test and build the YOLO dataset + data.yaml.

Reads ``<interim_dir>/manifest.csv`` (from convert_voc_to_yolo.py) and writes::

    <yolo_dir>/images/{train,val,test}/<uid>.<ext>
    <yolo_dir>/labels/{train,val,test}/<uid>.txt
    <yolo_dir>/data.yaml
    <yolo_dir>/split_summary.json
    <yolo_dir>/splits/{train,val,test}.txt

Leakage control (config ``split.strategy``): images in the same group always
land in the same split. ``random`` treats every image as its own group, which
is the documented default because RDD2022 India filenames carry no reliable
sequence/location metadata. See config.yaml for ``block`` and ``regex``.

Extra images (manifest ``source == "extra"``, from dataset.extra_dirs) are
always put in the training split. Only RDD2022 images are split, so val/test
are pure RDD2022 India and identical with or without extra data.

Usage:
    python src/split_dataset.py [--config config.yaml] [--force]
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import re
import shutil
import sys
from pathlib import Path
from typing import Any

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm

from src.common import get_class_names, load_config, resolve_path, set_seed, setup_logging
from src.convert_voc_to_yolo import LABELS_SUBDIR, MANIFEST_NAME, SOURCE_EXTRA

logger = logging.getLogger("split_dataset")

SPLITS = ("train", "val", "test")
RATIO_TOLERANCE = 1e-6
TRAILING_NUMBER = re.compile(r"^(.*?)(\d+)(?!.*\d)")


def validate_ratios(split_cfg: dict[str, Any]) -> dict[str, float]:
    """Return split ratios, checking they are non-negative and sum to 1."""
    ratios = {s: float(split_cfg.get(s, 0.0)) for s in SPLITS}
    if any(r < 0 for r in ratios.values()) or ratios["train"] <= 0 or ratios["val"] <= 0:
        raise ValueError(f"Invalid split ratios {ratios}: train and val must be > 0, test >= 0.")
    if abs(sum(ratios.values()) - 1.0) > RATIO_TOLERANCE:
        raise ValueError(f"Split ratios must sum to 1.0, got {sum(ratios.values()):.4f}: {ratios}")
    return ratios


def assign_groups(uids: pd.Series, split_cfg: dict[str, Any]) -> pd.Series:
    """Map every image uid to a group key according to split.strategy."""
    strategy = str(split_cfg.get("strategy", "random")).lower()
    if strategy == "random":
        return uids.copy()
    if strategy == "block":
        block_size = int(split_cfg.get("block_size", 1))

        def block_key(uid: str) -> str:
            m = TRAILING_NUMBER.match(uid)
            return f"{m.group(1)}#{int(m.group(2)) // block_size}" if m else uid

        return uids.map(block_key)
    if strategy == "regex":
        pattern = split_cfg.get("group_regex")
        if not pattern:
            raise ValueError("split.strategy is 'regex' but split.group_regex is not set.")
        compiled = re.compile(pattern)

        def regex_key(uid: str) -> str:
            m = compiled.search(uid)
            return m.group(1) if m and m.groups() else uid

        groups = uids.map(regex_key)
        unmatched = int((groups == uids).sum())
        if unmatched:
            logger.warning("%d images did not match group_regex and form singleton groups.", unmatched)
        return groups
    raise ValueError(f"Unknown split.strategy '{strategy}' (use random, block or regex).")


def _group_split(indices: np.ndarray, groups: np.ndarray, test_fraction: float, seed: int):
    """Split indices so that no group is shared; returns (keep, holdout)."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_fraction, random_state=seed)
    keep, holdout = next(splitter.split(indices, groups=groups))
    return indices[keep], indices[holdout]


def split_indices(groups: pd.Series, ratios: dict[str, float], seed: int) -> dict[str, np.ndarray]:
    """Return {split: row indices}; groups never cross splits.

    Note: GroupShuffleSplit applies ratios to the number of *groups*, so with
    grouped strategies the per-image proportions are approximate.
    """
    all_idx = np.arange(len(groups))
    group_arr = groups.to_numpy()
    if groups.nunique() < len(SPLITS):
        raise ValueError(f"Only {groups.nunique()} groups; need at least {len(SPLITS)} to split.")
    holdout_fraction = ratios["val"] + ratios["test"]
    train_idx, rest_idx = _group_split(all_idx, group_arr, holdout_fraction, seed)
    if ratios["test"] == 0:
        return {"train": train_idx, "val": rest_idx, "test": np.array([], dtype=int)}
    test_share = ratios["test"] / holdout_fraction
    val_idx, test_idx = _group_split(rest_idx, group_arr[rest_idx], test_share, seed)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def prepare_output(yolo_dir: Path, force: bool) -> None:
    """Refuse to overwrite existing processed data unless --force is given."""
    targets = [yolo_dir / "images", yolo_dir / "labels", yolo_dir / "splits"]
    existing = [t for t in targets if t.exists() and any(t.rglob("*"))]
    if existing and not force:
        raise FileExistsError(f"{yolo_dir} already contains a processed dataset. Re-run with --force to rebuild it.")
    for t in targets:
        if t.exists():
            shutil.rmtree(t)
    for split in SPLITS:
        (yolo_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (yolo_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
    (yolo_dir / "splits").mkdir(parents=True, exist_ok=True)


def place_file(src: Path, dst: Path, mode: str) -> str:
    """Copy, hardlink or symlink ``src`` to ``dst``; fall back to copying. Returns the mode used."""
    if mode in ("hardlink", "symlink"):
        try:
            if mode == "hardlink":
                os.link(src, dst)
            else:
                dst.symlink_to(src.resolve())
            return mode
        except OSError:
            pass
    shutil.copy2(src, dst)
    return "copy"


def write_split(rows: pd.DataFrame, split: str, labels_src: Path, yolo_dir: Path, mode: str) -> set[str]:
    """Place images and labels for one split; returns the set of link modes used."""
    used_modes: set[str] = set()
    for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=f"Writing {split}", unit="img"):
        image_src = Path(row.image_path)
        label_src = labels_src / f"{row.uid}.txt"
        if not image_src.is_file() or not label_src.is_file():
            raise FileNotFoundError(f"Missing source file for '{row.uid}': {image_src} / {label_src}. "
                                    "Re-run convert_voc_to_yolo.py --force.")
        used_modes.add(place_file(image_src, yolo_dir / "images" / split / f"{row.uid}{image_src.suffix.lower()}", mode))
        shutil.copy2(label_src, yolo_dir / "labels" / split / f"{row.uid}.txt")
    (yolo_dir / "splits" / f"{split}.txt").write_text("\n".join(rows["uid"]) + "\n", encoding="utf-8")
    return used_modes


def write_data_yaml(yolo_dir: Path, names: dict[int, str], has_test: bool) -> Path:
    """Write an Ultralytics data.yaml using forward-slash paths (valid on Windows and Linux)."""
    data = {"path": yolo_dir.resolve().as_posix(), "train": "images/train", "val": "images/val"}
    if has_test:
        data["test"] = "images/test"
    data["nc"] = len(names)
    data["names"] = {int(k): v for k, v in sorted(names.items())}
    out = yolo_dir / "data.yaml"
    with out.open("w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False)
    return out


def box_counts(frame: pd.DataFrame, names: dict[int, str]) -> dict[str, int]:
    return {names[cid]: int(frame.get(f"n_class_{cid}", pd.Series(dtype=int)).sum()) for cid in names}


def build_summary(manifest: pd.DataFrame, parts: dict[str, np.ndarray], groups: pd.Series,
                  names: dict[int, str], cfg: dict[str, Any], ratios: dict[str, float],
                  extra: pd.DataFrame) -> dict[str, Any]:
    """Collect per-split image/box counts for split_summary.json (RDD2022 rows + extra training rows)."""
    summary: dict[str, Any] = {
        "strategy": cfg["split"].get("strategy", "random"), "seed": int(cfg["split"]["seed"]),
        "ratios": ratios, "num_groups": int(groups.nunique()), "splits": {},
        "extra_train": {"images": int(len(extra)), "boxes": box_counts(extra, names)},
    }
    for split, idx in parts.items():
        sub = manifest.iloc[idx]
        boxes = box_counts(sub, names)
        summary["splits"][split] = {"images": int(len(sub)), "images_without_boxes": int((sub["num_boxes"] == 0).sum()),
                                    "groups": int(groups.iloc[idx].nunique()), "boxes": boxes}
    return summary


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--force", action="store_true", help="Delete and rebuild an existing YOLO dataset")
    args = parser.parse_args()

    setup_logging()
    cfg = load_config(args.config)
    interim_dir = resolve_path(cfg["dataset"]["interim_dir"])
    yolo_dir = resolve_path(cfg["dataset"]["yolo_dir"])
    manifest_path = interim_dir / MANIFEST_NAME
    names = get_class_names(cfg)
    seed = int(cfg["split"]["seed"])
    set_seed(seed)

    try:
        if not manifest_path.is_file():
            raise FileNotFoundError(f"{manifest_path} not found. Run src/convert_voc_to_yolo.py first.")
        full = pd.read_csv(manifest_path, dtype={"uid": str})
        is_extra = full["source"].eq(SOURCE_EXTRA) if "source" in full else pd.Series(False, index=full.index)
        manifest, extra = full[~is_extra].reset_index(drop=True), full[is_extra].reset_index(drop=True)
        if manifest.empty:
            raise ValueError(f"{manifest_path} has no RDD2022 images.")
        ratios = validate_ratios(cfg["split"])
        groups = assign_groups(manifest["uid"], cfg["split"])
        parts = split_indices(groups, ratios, seed)
        prepare_output(yolo_dir, args.force)
        modes: set[str] = set()
        for split, idx in parts.items():
            rows = manifest.iloc[idx]
            if split == "train" and not extra.empty:
                rows = pd.concat([rows, extra], ignore_index=True)
            modes |= write_split(rows, split, interim_dir / LABELS_SUBDIR, yolo_dir,
                                 str(cfg["split"].get("link_mode", "copy")))
    except (FileNotFoundError, FileExistsError, ValueError) as exc:
        logger.error("%s", exc)
        return 1

    data_yaml = write_data_yaml(yolo_dir, names, has_test=len(parts["test"]) > 0)
    summary = build_summary(manifest, parts, groups, names, cfg, ratios, extra)
    summary["file_placement"] = sorted(modes)
    (yolo_dir / "split_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    logger.info("Strategy: %s | seed: %d | groups: %d", summary["strategy"], seed, summary["num_groups"])
    if summary["strategy"] == "random":
        logger.info("Random per-image split: no grouping metadata used (see README, 'Data leakage').")
    for split, info in summary["splits"].items():
        logger.info("  %-5s images=%-6d empty=%-5d boxes=%s", split, info["images"],
                    info["images_without_boxes"], info["boxes"])
    if not extra.empty:
        logger.info("  + %d extra images added to train only, boxes=%s", len(extra), summary["extra_train"]["boxes"])
    logger.info("Wrote %s", data_yaml)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/train_detector.py
"""Fine-tune a pretrained Ultralytics YOLO detector on the prepared RDD2022 data.

All defaults come from config.yaml; common settings can be overridden:

    python src/train_detector.py
    python src/train_detector.py --model yolo11s.pt --epochs 150 --imgsz 800 --batch 8
    python src/train_detector.py --device cpu --epochs 1     # quick CPU smoke run
    python src/train_detector.py --resume                     # continue the newest interrupted run
"""

from __future__ import annotations

import argparse
import json
import logging
import platform
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from src.common import load_config, resolve_device, resolve_path, set_seed, setup_logging

logger = logging.getLogger("train_detector")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--model", default=None, help="Pretrained weights, e.g. yolo11n.pt or yolo11s.pt")
    parser.add_argument("--epochs", type=int, default=None)
    parser.add_argument("--batch", type=int, default=None, help="Batch size (-1 = Ultralytics auto-batch on GPU)")
    parser.add_argument("--imgsz", "--image-size", dest="imgsz", type=int, default=None)
    parser.add_argument("--device", default=None, help="auto | cpu | 0 | 0,1")
    parser.add_argument("--workers", type=int, default=None)
    parser.add_argument("--patience", type=int, default=None)
    parser.add_argument("--name", default=None, help="Run name under runs/detect/")
    parser.add_argument("--resume", nargs="?", const="latest", default=None,
                        help="Resume an interrupted run: path to its last.pt, or no value for the newest run")
    return parser.parse_args()


def build_train_params(cfg: dict[str, Any], args: argparse.Namespace) -> dict[str, Any]:
    """Merge config values with command-line overrides."""
    t = cfg["training"]

    def pick(cli_value, cfg_value):
        return cfg_value if cli_value is None else cli_value

    return {
        "model": pick(args.model, cfg["model"]["name"]),
        "epochs": int(pick(args.epochs, t["epochs"])),
        "batch": int(pick(args.batch, t["batch"])),
        "imgsz": int(pick(args.imgsz, t["image_size"])),
        "device": resolve_device(pick(args.device, t.get("device", "auto"))),
        "workers": int(pick(args.workers, t["workers"])),
        "patience": int(pick(args.patience, t["patience"])),
        "seed": int(t["seed"]),
        "deterministic": bool(t.get("deterministic", True)),
        "name": pick(args.name, t.get("run_name", "train")),
    }


def find_resume_checkpoint(runs_dir: Path, resume: str) -> Path:
    """Return the last.pt to resume from: an explicit path, or the newest one under runs_dir."""
    if resume != "latest":
        path = resolve_path(resume)
        if path is None or not path.is_file():
            raise FileNotFoundError(f"Checkpoint to resume not found: {path}")
        return path
    candidates = sorted(runs_dir.glob("*/weights/last.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No runs/*/weights/last.pt found under {runs_dir}; nothing to resume.")
    return candidates[0]


def check_dataset(data_yaml: Path) -> dict[str, Any]:
    """Make sure the YOLO dataset exists and return the split summary if present."""
    if not data_yaml.is_file():
        raise FileNotFoundError(
            f"{data_yaml} not found. Prepare the dataset first:\n"
            "  python src/convert_voc_to_yolo.py\n  python src/split_dataset.py"
        )
    for split in ("train", "val"):
        folder = data_yaml.parent / "images" / split
        if not folder.is_dir() or not any(folder.iterdir()):
            raise FileNotFoundError(f"No images in {folder}. Re-run src/split_dataset.py.")
    summary_path = data_yaml.parent / "split_summary.json"
    return json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.is_file() else {}


def record_run(save_dir: Path, params: dict[str, Any], data_yaml: Path, split_summary: dict[str, Any]) -> None:
    """Write run_metadata.json next to the Ultralytics outputs."""
    import torch
    import ultralytics

    meta = {
        "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "params": params,
        "data_yaml": data_yaml.as_posix(),
        "dataset_split": split_summary,
        "versions": {"python": platform.python_version(), "torch": torch.__version__,
                     "ultralytics": ultralytics.__version__},
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    (save_dir / "run_metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")


def main() -> int:
    args = parse_args()
    setup_logging()
    cfg = load_config(args.config)
    data_yaml = resolve_path(cfg["dataset"]["yolo_dir"]) / "data.yaml"
    runs_dir = resolve_path(cfg["paths"]["runs_dir"])

    try:
        split_summary = check_dataset(data_yaml)
    except FileNotFoundError as exc:
        logger.error("%s", exc)
        return 1

    try:
        from ultralytics import YOLO
    except ImportError:
        logger.error("Ultralytics is not installed. Run: pip install -r requirements.txt")
        return 1

    if args.resume:
        try:
            checkpoint = find_resume_checkpoint(runs_dir, args.resume)
        except FileNotFoundError as exc:
            logger.error("%s", exc)
            return 1
        logger.info("Resuming training from %s (original settings are kept)", checkpoint)
        model = YOLO(str(checkpoint))
        model.train(resume=True)
        trainer_args = model.trainer.args
        params = {k: getattr(trainer_args, k, None)
                  for k in ("model", "epochs", "batch", "imgsz", "device", "workers", "patience", "seed", "name")}
        params["resumed_from"] = checkpoint.as_posix()
    else:
        params = build_train_params(cfg, args)
        set_seed(params["seed"])
        logger.info("Training configuration:")
        for key, value in params.items():
            logger.info("  %-13s %s", key, value)
        if params["device"] == "cpu":
            logger.warning("Training on CPU - this will be slow. Install a CUDA build of PyTorch to use a GPU.")
        model_name = params.pop("model")
        model = YOLO(model_name)  # pretrained checkpoint; Ultralytics downloads official weights if missing
        model.train(data=data_yaml.as_posix(), project=runs_dir.as_posix(), pretrained=True, **params)
        params["model"] = model_name

    save_dir = Path(model.trainer.save_dir)
    record_run(save_dir, params, data_yaml, split_summary)
    best = save_dir / "weights" / "best.pt"
    if not best.is_file():
        logger.error("Training finished but %s was not produced.", best)
        return 1

    export_to = resolve_path(cfg["training"].get("export_weights_to"))
    if export_to:
        export_to.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(best, export_to)
        logger.info("Copied best weights to %s", export_to)
    logger.info("Run directory: %s", save_dir)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/train_severity.py
"""Train the Low / Medium / High severity classifier on human-labeled defect crops.

Inputs : data/severity_crops/metadata.csv  (src/crop_defects.py --save-crops)
         data/severity_crops/labels.csv    (streamlit run label_severity.py)
Outputs: models/severity_classifier.pt     (picked up automatically by the app)
         reports/severity/                 (metrics, confusion matrix, heuristic comparison)

The validation split is grouped by source image (crops of one photo never
appear in both splits) and stratified by severity level.

Usage:
    python src/train_severity.py [--epochs 30] [--arch mobilenet_v3_small] [--device cpu]
"""

from __future__ import annotations

import argparse
import json
import logging
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from src.common import load_config, resolve_device, resolve_path, set_seed, setup_logging
from src.crop_defects import METADATA_NAME
from src.severity import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    SEVERITY_LEVELS,
    HeuristicSeverityEstimator,
    build_classifier,
    preprocess_crop,
    torch_device,
)

logger = logging.getLogger("train_severity")

BRIGHTNESS_JITTER = 0.15   # +-15% brightness augmentation
CONTRAST_JITTER = 0.15     # +-15% contrast augmentation
HFLIP_PROB = 0.5


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--epochs", type=int, default=None)
    parser.add_argument("--batch", type=int, default=None)
    parser.add_argument("--arch", default=None, help="mobilenet_v3_small | mobilenet_v3_large | resnet18")
    parser.add_argument("--device", default=None, help="auto | cpu | 0")
    return parser.parse_args()


def load_labeled_crops(cfg: dict[str, Any]) -> pd.DataFrame:
    """Join crop metadata with human labels; keep Low/Medium/High only."""
    metadata_path = resolve_path(cfg["severity"]["crops_dir"]) / METADATA_NAME
    labels_path = resolve_path(cfg["severity"]["training"]["labels_file"])
    for path, hint in ((metadata_path, "python src/crop_defects.py ... --save-crops"),
                       (labels_path, "streamlit run label_severity.py")):
        if not path.is_file():
            raise FileNotFoundError(f"{path} not found. Create it with: {hint}")
    labels = pd.read_csv(labels_path)[["crop_path", "severity"]]
    data = pd.read_csv(metadata_path).merge(labels, on="crop_path", how="inner")
    data = data[data["severity"].isin(SEVERITY_LEVELS)].reset_index(drop=True)
    data["crop_file"] = data["crop_path"].map(resolve_path)
    missing = ~data["crop_file"].map(lambda p: p.is_file())
    if missing.any():
        logger.warning("%d labeled crops are missing on disk and are skipped.", int(missing.sum()))
        data = data[~missing].reset_index(drop=True)
    data["target"] = data["severity"].map({lvl: i for i, lvl in enumerate(SEVERITY_LEVELS)})
    return data


def check_counts(data: pd.DataFrame, min_per_level: int) -> None:
    counts = data["severity"].value_counts().reindex(SEVERITY_LEVELS, fill_value=0)
    logger.info("Labeled crops per level: %s", counts.to_dict())
    too_few = counts[counts < min_per_level]
    if not too_few.empty:
        raise ValueError(f"Need at least {min_per_level} labeled crops per level; too few for: "
                         f"{too_few.to_dict()}. Label more with `streamlit run label_severity.py`.")


def split_train_val(data: pd.DataFrame, val_fraction: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Stratified (by level), grouped (by source image) split."""
    n_splits = max(2, round(1 / val_fraction))
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    train_idx, val_idx = next(splitter.split(data, data["target"], groups=data["image_name"]))
    return train_idx, val_idx


class CropDataset:
    """Minimal torch-compatible dataset of labeled crops."""

    def __init__(self, frame: pd.DataFrame, image_size: int, augment: bool, seed: int) -> None:
        self.paths = frame["crop_file"].tolist()
        self.targets = frame["target"].tolist()
        self.image_size = image_size
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self) -> int:
        return len(self.paths)

    def _augment(self, crop: np.ndarray) -> np.ndarray:
        if self.rng.random() < HFLIP_PROB:
            crop = crop[:, ::-1]
        alpha = 1.0 + self.rng.uniform(-CONTRAST_JITTER, CONTRAST_JITTER)
        beta = 255.0 * self.rng.uniform(-BRIGHTNESS_JITTER, BRIGHTNESS_JITTER)
        return np.clip(crop.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

    def __getitem__(self, idx: int):
        import torch

        crop = cv2.imread(str(self.paths[idx]))
        if self.augment:
            crop = self._augment(crop)
        x = torch.from_numpy(np.ascontiguousarray(preprocess_crop(crop, self.image_size)))
        return x, self.targets[idx]


def class_weights(targets: np.ndarray) -> np.ndarray:
    """Inverse-frequency weights so rare levels are not ignored."""
    counts = np.bincount(targets, minlength=len(SEVERITY_LEVELS)).astype(np.float32)
    return counts.sum() / (len(SEVERITY_LEVELS) * np.maximum(counts, 1.0))


def predict_all(model, loader, device: str) -> np.ndarray:
    import torch

    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in loader:
            preds.append(model(x.to(device)).argmax(dim=1).cpu().numpy())
    return np.concatenate(preds) if preds else np.array([], dtype=int)


def train_model(train_ds: CropDataset, val_ds: CropDataset, val_targets: np.ndarray, tcfg: dict[str, Any],
                arch: str, device: str) -> tuple[Any, float, int]:
    """Fine-tune an ImageNet-pretrained classifier; return (best_state_dict, best_macro_f1, best_epoch)."""
    import torch
    from torch.utils.data import DataLoader

    try:
        model = build_classifier(arch, len(SEVERITY_LEVELS), pretrained=True)
    except Exception as exc:  # e.g. no internet to download ImageNet weights
        logger.warning("Could not load ImageNet weights (%s); training from random initialisation.", exc)
        model = build_classifier(arch, len(SEVERITY_LEVELS), pretrained=False)
    model.to(device)
    train_loader = DataLoader(train_ds, batch_size=int(tcfg["batch"]), shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=int(tcfg["batch"]), shuffle=False, num_workers=0)
    weights = torch.tensor(class_weights(np.array(train_ds.targets)), device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(tcfg["learning_rate"]),
                                  weight_decay=float(tcfg["weight_decay"]))

    best_f1, best_epoch, best_state, stale = -1.0, 0, None, 0
    for epoch in range(1, int(tcfg["epochs"]) + 1):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(x.to(device)), torch.as_tensor(y, device=device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
        val_f1 = f1_score(val_targets, predict_all(model, val_loader, device), average="macro", zero_division=0)
        logger.info("epoch %3d  train_loss %.4f  val_macro_f1 %.3f", epoch, total_loss / len(train_ds), val_f1)
        if val_f1 > best_f1:
            best_f1, best_epoch, stale = val_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= int(tcfg["patience"]):
                logger.info("Early stopping: no improvement for %d epochs.", stale)
                break
    return best_state, best_f1, best_epoch


def heuristic_predictions(frame: pd.DataFrame, cfg: dict[str, Any]) -> np.ndarray:
    """Heuristic estimate for each crop (to compare it against the human labels)."""
    estimator = HeuristicSeverityEstimator(cfg["severity"]["heuristic"])
    preds = []
    for row in frame.itertuples(index=False):
        det = {"class_name": row.class_name, "bbox": [row.x1, row.y1, row.x2, row.y2]}
        result = estimator.predict(cv2.imread(str(row.crop_file)), det, (row.image_height, row.image_width))
        preds.append(SEVERITY_LEVELS.index(result["severity"]))
    return np.array(preds)


def save_confusion_matrix(cm: np.ndarray, title: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(SEVERITY_LEVELS)), SEVERITY_LEVELS)
    ax.set_yticks(range(len(SEVERITY_LEVELS)), SEVERITY_LEVELS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Human label")
    ax.set_title(title, loc="left")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def evaluate(name: str, y_true: np.ndarray, y_pred: np.ndarray, out_dir: Path) -> dict[str, Any]:
    labels = list(range(len(SEVERITY_LEVELS)))
    report = classification_report(y_true, y_pred, labels=labels, target_names=list(SEVERITY_LEVELS),
                                   output_dict=True, zero_division=0)
    logger.info("%s on validation crops:\n%s", name, classification_report(
        y_true, y_pred, labels=labels, target_names=list(SEVERITY_LEVELS), zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    save_confusion_matrix(cm, f"{name} - validation", out_dir / f"confusion_matrix_{name.lower()}.png")
    return {"accuracy": report["accuracy"], "macro_f1": report["macro avg"]["f1-score"],
            "per_level": {lvl: report[lvl] for lvl in SEVERITY_LEVELS}, "confusion_matrix": cm.tolist()}


def main() -> int:
    args = parse_args()
    setup_logging()
    cfg = load_config(args.config)
    tcfg = dict(cfg["severity"]["training"])
    tcfg["epochs"] = args.epochs or tcfg["epochs"]
    tcfg["batch"] = args.batch or tcfg["batch"]
    arch = args.arch or tcfg["arch"]
    seed = int(tcfg["seed"])
    set_seed(seed)

    try:
        data = load_labeled_crops(cfg)
        check_counts(data, int(tcfg["min_per_level"]))
        train_idx, val_idx = split_train_val(data, float(tcfg["val_fraction"]), seed)
    except (FileNotFoundError, ValueError) as exc:
        logger.error("%s", exc)
        return 1
    train_df, val_df = data.iloc[train_idx], data.iloc[val_idx]
    logger.info("Train crops: %d | validation crops: %d (grouped by source image)", len(train_df), len(val_df))

    device = torch_device(resolve_device(args.device or cfg["training"].get("device", "auto")))
    size = int(tcfg["image_size"])
    val_targets = val_df["target"].to_numpy()
    state, best_f1, best_epoch = train_model(CropDataset(train_df, size, True, seed),
                                             CropDataset(val_df, size, False, seed), val_targets, tcfg, arch, device)

    out_path = resolve_path(cfg["severity"]["model_path"])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    import torch

    torch.save({"arch": arch, "state_dict": state, "labels": list(SEVERITY_LEVELS), "image_size": size,
                "mean": list(IMAGENET_MEAN), "std": list(IMAGENET_STD), "best_epoch": best_epoch,
                "val_macro_f1": best_f1, "num_train": len(train_df), "num_val": len(val_df),
                "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, out_path)
    logger.info("Saved severity model to %s (best epoch %d, val macro-F1 %.3f)", out_path, best_epoch, best_f1)

    from torch.utils.data import DataLoader

    model = build_classifier(arch, len(SEVERITY_LEVELS))
    model.load_state_dict(state)
    model.to(device)
    val_loader = DataLoader(CropDataset(val_df, size, False, seed), batch_size=int(tcfg["batch"]), num_workers=0)
    reports_dir = resolve_path(cfg["paths"]["reports_dir"]) / "severity"
    reports_dir.mkdir(parents=True, exist_ok=True)
    results = {
        "model": evaluate("Classifier", val_targets, predict_all(model, val_loader, device), reports_dir),
        "heuristic": evaluate("Heuristic", val_targets, heuristic_predictions(val_df, cfg), reports_dir),
        "num_train": len(train_df), "num_val": len(val_df), "arch": arch, "best_epoch": best_epoch,
        "label_counts": data["severity"].value_counts().to_dict(),
    }
    (reports_dir / "metrics.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
    logger.info("Validation macro-F1: classifier %.3f vs heuristic %.3f. Reports in %s",
                results["model"]["macro_f1"], results["heuristic"]["macro_f1"], reports_dir)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/visualize_annotations.py
"""Visually check the RDD2022 annotations (and our conversion of them).

For a random sample of boxes per class this writes to reports/annotation_samples/:

    <class>/sheet.jpg        contact sheet: one tile per box (crop with margin + id)
    <class>/<image>.jpg      full image with ALL of its original XML boxes drawn
    review.csv               one row per sampled box - fill in the 'verdict' column
    box_size_stats.csv       box size statistics per class (all boxes, not just the sample)

It also checks two things that would make labels look wrong even though the
XML is correct:
  * EXIF rotation - photos with an EXIF orientation tag are shown rotated by
    OpenCV/YOLO, while annotation tools often ignore the tag, so boxes shift.
  * Conversion   - if data/interim/labels exists, our YOLO boxes are drawn as
    thin white rectangles; they must sit exactly on the coloured XML boxes.

Usage:
    python src/visualize_annotations.py                       # 40 boxes per class
    python src/visualize_annotations.py --classes D40 --n 80  # 80 potholes
"""

from __future__ import annotations

import argparse
import logging
import random
import sys
from collections import defaultdict
from pathlib import Path
from typing import Any

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from src.common import (
    get_class_names,
    get_image_extensions,
    get_source_code_map,
    load_config,
    resolve_path,
    setup_logging,
    to_project_relative,
)
from src.inference import CLASS_COLORS_BGR, draw_predictions
from src.voc_dataset import VocParseError, discover_dataset, parse_voc_xml

logger = logging.getLogger("visualize_annotations")

EXIF_ORIENTATION_TAG = 274
TILE_SIZE = 220
TILE_MARGIN = 0.35            # context around each box in a tile (fraction of box size)
SHEET_COLUMNS = 8
TILE_LABEL_HEIGHT = 22
TINY_BOX_PX = 32              # boxes smaller than this (shorter side) are "tiny"
CONVERTED_COLOR = (255, 255, 255)


def exif_orientation(image_path: Path) -> int:
    """EXIF orientation tag (1 = normal); 1 if absent or unreadable."""
    try:
        with Image.open(image_path) as img:
            return int(img.getexif().get(EXIF_ORIENTATION_TAG, 1))
    except (OSError, ValueError):
        return 1


def collect_boxes(pairs, code_map: dict[str, int], names: dict[int, str]) -> tuple[list[dict], list[str]]:
    """All target-class boxes from the XML files: one record per box."""
    boxes, problems = [], []
    for pair in tqdm(pairs, desc="Reading XML", unit="file"):
        try:
            ann, _ = parse_voc_xml(pair.xml_path)
        except VocParseError as exc:
            problems.append(f"{pair.xml_path.name}: {exc}")
            continue
        for idx, obj in enumerate(ann.objects, start=1):
            code = obj.name.strip().upper()
            if code not in code_map:
                continue
            x1, x2 = sorted((obj.xmin, obj.xmax))  # some XML boxes have swapped min/max
            y1, y2 = sorted((obj.ymin, obj.ymax))
            boxes.append({"uid": pair.uid, "image_path": pair.image_path, "box_index": idx, "code": code,
                          "class_id": code_map[code], "class_name": names[code_map[code]],
                          "bbox": [int(round(v)) for v in (x1, y1, x2, y2)],
                          "xml_width": ann.width, "xml_height": ann.height})
    return boxes, problems


def size_stats(boxes: list[dict]) -> pd.DataFrame:
    """Per-class box size statistics (pixels)."""
    rows = []
    frame = pd.DataFrame([{"class_name": b["class_name"], "w": b["bbox"][2] - b["bbox"][0],
                           "h": b["bbox"][3] - b["bbox"][1]} for b in boxes])
    if frame.empty:
        return frame
    for name, g in frame.groupby("class_name"):
        short = g[["w", "h"]].min(axis=1)
        rows.append({"class_name": name, "boxes": len(g), "median_width_px": int(g.w.median()),
                     "median_height_px": int(g.h.median()),
                     f"tiny_boxes_pct(<{TINY_BOX_PX}px)": round(100 * (short < TINY_BOX_PX).mean(), 1),
                     "largest_box_px": f"{int(g.w.max())}x{int(g.h.max())}"})
    return pd.DataFrame(rows)


def read_converted(labels_dir: Path | None, uid: str, width: int, height: int) -> list[list[int]]:
    """Our converted YOLO boxes for an image, in pixels ([] if not available)."""
    if labels_dir is None:
        return []
    path = labels_dir / f"{uid}.txt"
    if not path.is_file():
        return []
    out = []
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) == 5:
            cx, cy, w, h = (float(v) for v in parts[1:])
            out.append([int(round((cx - w / 2) * width)), int(round((cy - h / 2) * height)),
                        int(round((cx + w / 2) * width)), int(round((cy + h / 2) * height))])
    return out


def draw_full_image(image: np.ndarray, image_boxes: list[dict], converted: list[list[int]]) -> np.ndarray:
    """All XML boxes (coloured, labelled with their id) plus converted boxes (thin white)."""
    dets = [{"class_id": b["class_id"], "class_name": f"#{b['box_index']} {b['class_name']}", "confidence": 1.0,
             "bbox": b["bbox"]} for b in image_boxes]
    canvas = draw_predictions(image, dets, box_thickness=3, show_confidence=False)
    for x1, y1, x2, y2 in converted:
        cv2.rectangle(canvas, (x1, y1), (x2, y2), CONVERTED_COLOR, 1, cv2.LINE_AA)
    return canvas


def make_tile(image: np.ndarray, box: dict, tile_id: str) -> np.ndarray:
    """Square tile: the box with some context, box outlined, id written underneath."""
    h, w = image.shape[:2]
    x1, y1, x2, y2 = box["bbox"]
    mx, my = int((x2 - x1) * TILE_MARGIN) + 8, int((y2 - y1) * TILE_MARGIN) + 8
    left, top = min(max(0, x1 - mx), w - 1), min(max(0, y1 - my), h - 1)
    right, bottom = max(min(w, x2 + mx), left + 1), max(min(h, y2 + my), top + 1)
    crop = np.ascontiguousarray(image[top:bottom, left:right])
    color = CLASS_COLORS_BGR.get(box["class_id"], (128, 128, 128))
    cv2.rectangle(crop, (x1 - left, y1 - top), (x2 - left, y2 - top), color, 2)
    scale = TILE_SIZE / max(crop.shape[:2]) if crop.size else 1.0
    crop = cv2.resize(crop, (max(1, int(crop.shape[1] * scale)), max(1, int(crop.shape[0] * scale))))
    tile = np.full((TILE_SIZE + TILE_LABEL_HEIGHT, TILE_SIZE, 3), 30, np.uint8)
    oy, ox = (TILE_SIZE - crop.shape[0]) // 2, (TILE_SIZE - crop.shape[1]) // 2
    tile[oy:oy + crop.shape[0], ox:ox + crop.shape[1]] = crop
    cv2.putText(tile, f"{tile_id}  {x2 - x1}x{y2 - y1}px", (4, TILE_SIZE + 16), cv2.FONT_HERSHEY_SIMPLEX,
                0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return tile


def make_sheet(tiles: list[np.ndarray]) -> np.ndarray:
    rows = [tiles[i:i + SHEET_COLUMNS] for i in range(0, len(tiles), SHEET_COLUMNS)]
    blank = np.full_like(tiles[0], 30)
    return np.vstack([np.hstack(r + [blank] * (SHEET_COLUMNS - len(r))) for r in rows])


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", default=None, help="Path to config.yaml")
    parser.add_argument("--raw-dir", default=None, help="Override dataset.raw_dir")
    parser.add_argument("--classes", nargs="*", default=None, help="Codes or names, e.g. D40 pothole (default: all)")
    parser.add_argument("--n", type=int, default=40, help="Boxes sampled per class")
    parser.add_argument("--seed", type=int, default=None, help="Sampling seed (default: split.seed)")
    parser.add_argument("--out", default=None, help="Output folder (default: reports/annotation_samples)")
    args = parser.parse_args()

    setup_logging()
    cfg = load_config(args.config)
    names, code_map = get_class_names(cfg), get_source_code_map(cfg)
    raw_dir = resolve_path(args.raw_dir or cfg["dataset"]["raw_dir"])
    out_dir = resolve_path(args.out) if args.out else resolve_path(cfg["paths"]["reports_dir"]) / "annotation_samples"
    interim_labels = resolve_path(cfg["dataset"]["interim_dir"]) / "labels"
    labels_dir = interim_labels if interim_labels.is_dir() else None
    wanted = {c.upper() for c in args.classes} if args.classes else None

    try:
        result = discover_dataset(raw_dir, get_image_extensions(cfg), cfg["dataset"]["annotation_extension"])
    except FileNotFoundError as exc:
        logger.error("%s", exc)
        return 1
    if not result.pairs:
        logger.error("No image/XML pairs under %s. Run src/inspect_dataset.py to see what was found.", raw_dir)
        return 1

    rotated = [p for p in tqdm(result.pairs, desc="Checking EXIF", unit="img") if exif_orientation(p.image_path) != 1]
    boxes, problems = collect_boxes(result.pairs, code_map, names)
    stats = size_stats(boxes)
    out_dir.mkdir(parents=True, exist_ok=True)
    stats.to_csv(out_dir / "box_size_stats.csv", index=False)

    by_class: dict[str, list[dict]] = defaultdict(list)
    for b in boxes:
        if wanted is None or b["code"] in wanted or b["class_name"].upper() in wanted:
            by_class[b["class_name"]].append(b)
    rng = random.Random(args.seed if args.seed is not None else int(cfg["split"]["seed"]))
    boxes_by_uid: dict[str, list[dict]] = defaultdict(list)
    for b in boxes:
        boxes_by_uid[b["uid"]].append(b)

    review_rows, mismatches = [], 0
    for class_name in sorted(by_class):
        sample = rng.sample(by_class[class_name], min(args.n, len(by_class[class_name])))
        class_dir = out_dir / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        tiles = []
        for i, box in enumerate(tqdm(sample, desc=f"Drawing {class_name}", unit="box"), start=1):
            image = cv2.imread(str(box["image_path"]))
            if image is None:
                continue
            h, w = image.shape[:2]
            tile_id = f"{class_name[:3].upper()}-{i:03d}"
            tiles.append(make_tile(image, box, tile_id))
            converted = read_converted(labels_dir, box["uid"], w, h)
            clipped = np.clip(box["bbox"], 0, [w, h, w, h])  # the converter clips boxes to the image
            if labels_dir and converted and not any(np.abs(np.subtract(c, clipped)).max() <= 2 for c in converted):
                mismatches += 1
            full_path = class_dir / f"{box['uid']}.jpg"
            cv2.imwrite(str(full_path), draw_full_image(image, boxes_by_uid[box["uid"]], converted))
            x1, y1, x2, y2 = box["bbox"]
            review_rows.append({"tile_id": tile_id, "class_name": class_name, "code": box["code"],
                                "image_name": box["image_path"].name, "box_index": box["box_index"],
                                "x1": x1, "y1": y1, "x2": x2, "y2": y2, "width_px": x2 - x1, "height_px": y2 - y1,
                                "full_image": to_project_relative(full_path), "verdict": "", "comment": ""})
        if tiles:
            cv2.imwrite(str(class_dir / "sheet.jpg"), make_sheet(tiles))
    pd.DataFrame(review_rows).to_csv(out_dir / "review.csv", index=False)

    logger.info("=" * 70)
    logger.info("Image/XML pairs: %d | target-class boxes: %d", len(result.pairs), len(boxes))
    logger.info("Box sizes (all boxes):\n%s", stats.to_string(index=False) if not stats.empty else "(none)")
    if rotated:
        logger.warning("EXIF rotation: %d images have an orientation tag != 1 (e.g. %s). Their boxes may be "
                       "shifted/rotated - check their full images carefully.", len(rotated), rotated[0].image_path.name)
    else:
        logger.info("EXIF rotation: none found - boxes are not affected by photo rotation.")
    if labels_dir is None:
        logger.info("Conversion check skipped (run src/convert_voc_to_yolo.py first to include it).")
    elif mismatches:
        logger.warning("Conversion check: %d sampled boxes have NO matching converted YOLO box.", mismatches)
    else:
        logger.info("Conversion check: every sampled XML box has a matching converted YOLO box.")
    if problems:
        logger.warning("%d XML files could not be read (e.g. %s)", len(problems), problems[0])
    logger.info("Open %s: start with <class>/sheet.jpg, then fill the 'verdict' column in review.csv "
                "(ok / wrong_class / bad_box / not_damage).", out_dir)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/proj/src/voc_dataset.py
"""Discovery of images / Pascal VOC XML files and XML parsing.

The RDD2022 download layout varies between releases (e.g. ``India/train/images``
and ``India/train/annotations/xmls``), so nothing here assumes a fixed folder
structure: files are found recursively and images are matched to XML files by
filename stem.
"""

from __future__ import annotations

import logging
import re
import xml.etree.ElementTree as ET
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path

from PIL import Image

logger = logging.getLogger(__name__)


@dataclass
class VocObject:
    """One annotated object from a VOC XML file (pixel coordinates)."""

    name: str
    xmin: float
    ymin: float
    xmax: float
    ymax: float


@dataclass
class VocAnnotation:
    """Parsed content of one VOC XML file."""

    xml_path: Path
    filename: str | None
    width: int | None
    height: int | None
    objects: list[VocObject] = field(default_factory=list)


@dataclass
class ImageXmlPair:
    """An image matched with its XML annotation, plus a dataset-unique id."""

    uid: str
    image_path: Path
    xml_path: Path


@dataclass
class DiscoveryResult:
    """Everything found while scanning the raw dataset directory."""

    images: list[Path]
    xmls: list[Path]
    pairs: list[ImageXmlPair]
    unmatched_images: list[Path]
    unmatched_xmls: list[Path]


class VocParseError(ValueError):
    """Raised when an XML file cannot be parsed as a VOC annotation."""


def find_files(root: Path, extensions: set[str]) -> list[Path]:
    """Recursively find files under ``root`` whose suffix is in ``extensions``."""
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in extensions)


def _common_prefix_len(a: Path, b: Path) -> int:
    count = 0
    for x, y in zip(a.parts, b.parts):
        if x != y:
            break
        count += 1
    return count


def _make_uid(image_path: Path, root: Path, duplicated: bool, prefix: str = "") -> str:
    """Stem for unique names; prefix with the relative folder if the stem is duplicated.

    With a ``prefix`` (used for extra datasets) the uid is always prefix + relative path,
    so extra images can never collide with RDD2022 file names.
    """
    if not duplicated and not prefix:
        return image_path.stem
    rel = image_path.relative_to(root).with_suffix("")
    return prefix + re.sub(r"[^A-Za-z0-9_.-]+", "__", "__".join(rel.parts))


def match_images_to_xmls(root: Path, images: list[Path], xmls: list[Path], uid_prefix: str = "") -> DiscoveryResult:
    """Pair images and XML files by filename stem.

    * An XML named after the full image file name (``images.jpeg.xml``) wins; this is
      how two images with the same stem but different extensions are told apart.
    * If the image's own folder contains XML files (XML-next-to-image layout), only
      XML files from that folder are considered - never a same-named file elsewhere.
    * Otherwise (e.g. RDD2022: images/ + annotations/xmls/) the XML closest in the
      folder tree (longest common path prefix) is chosen.
    """
    xml_by_stem: dict[str, list[Path]] = defaultdict(list)
    dirs_with_xml: set[Path] = set()
    for x in xmls:
        xml_by_stem[x.stem].append(x)
        dirs_with_xml.add(x.parent)
    image_stem_counts: dict[str, int] = defaultdict(int)
    for img in images:
        image_stem_counts[img.stem] += 1

    pairs: list[ImageXmlPair] = []
    unmatched_images: list[Path] = []
    used_xmls: set[Path] = set()
    for img in images:
        candidates = [x for x in xml_by_stem.get(img.name, []) if x not in used_xmls and x.parent == img.parent]
        if not candidates:
            candidates = [x for x in xml_by_stem.get(img.stem, []) if x not in used_xmls]
            if img.parent in dirs_with_xml:
                candidates = [x for x in candidates if x.parent == img.parent]
        if not candidates:
            unmatched_images.append(img)
            continue
        xml = max(candidates, key=lambda x: _common_prefix_len(x.parent, img.parent))
        used_xmls.add(xml)
        uid = _make_uid(img, root, image_stem_counts[img.stem] > 1, uid_prefix)
        pairs.append(ImageXmlPair(uid=uid, image_path=img, xml_path=xml))

    unmatched_xmls = [x for x in xmls if x not in used_xmls]
    return DiscoveryResult(images, xmls, pairs, unmatched_images, unmatched_xmls)


def discover_dataset(root: Path, image_extensions: set[str], annotation_extension: str = ".xml",
                     uid_prefix: str = "") -> DiscoveryResult:
    """Scan ``root`` recursively and match images to XML annotations."""
    if not root.is_dir():
        raise FileNotFoundError(
            f"Raw dataset directory not found: {root}\n"
            "Download RDD2022 India and extract it there (or change dataset.raw_dir in config.yaml)."
        )
    images = find_files(root, image_extensions)
    xmls = find_files(root, {annotation_extension.lower()})
    return match_images_to_xmls(root, images, xmls, uid_prefix)


def _find_float(node: ET.Element, tag: str) -> float:
    text = node.findtext(tag)
    if text is None or text.strip() == "":
        raise VocParseError(f"missing <{tag}>")
    try:
        return float(text.strip())
    except ValueError as exc:
        raise VocParseError(f"non-numeric <{tag}>: {text!r}") from exc


def _parse_size(root: ET.Element) -> tuple[int | None, int | None]:
    size = root.find("size")
    if size is None:
        return None, None
    try:
        width = int(float(size.findtext("width", "0")))
        height = int(float(size.findtext("height", "0")))
    except ValueError:
        return None, None
    return (width or None), (height or None)


def parse_voc_xml(xml_path: Path) -> tuple[VocAnnotation, list[str]]:
    """Parse a VOC XML file.

    Returns the annotation and a list of per-object problems (objects that could
    not be read are skipped but reported). Raises VocParseError if the file as a
    whole is unreadable.
    """
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError as exc:
        raise VocParseError(f"malformed XML: {exc}") from exc

    width, height = _parse_size(root)
    annotation = VocAnnotation(xml_path=xml_path, filename=root.findtext("filename"), width=width, height=height)
    problems: list[str] = []
    for idx, obj in enumerate(root.findall("object")):
        name = (obj.findtext("name") or "").strip()
        box = obj.find("bndbox")
        if not name or box is None:
            problems.append(f"object {idx}: missing <name> or <bndbox>")
            continue
        try:
            coords = [_find_float(box, t) for t in ("xmin", "ymin", "xmax", "ymax")]
        except VocParseError as exc:
            problems.append(f"object {idx} ({name}): {exc}")
            continue
        annotation.objects.append(VocObject(name, *coords))
    return annotation, problems


EXIF_ORIENTATION_TAG = 274
EXIF_ROTATED_90 = {5, 6, 7, 8}  # orientations that swap width and height


def read_image_size(image_path: Path) -> tuple[int, int]:
    """Return the displayed (width, height) of an image without decoding all pixels.

    EXIF rotation is taken into account, because OpenCV/YOLO load images upright.
    """
    with Image.open(image_path) as img:
        width, height = img.size
        if img.getexif().get(EXIF_ORIENTATION_TAG, 1) in EXIF_ROTATED_90:
            return height, width
        return width, height


In [ ]:
# Point outputs at Google Drive so they survive a disconnect.
import yaml

cfg_path = PROJ / "config.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
cfg["paths"]["runs_dir"] = f"{OUTPUT_DIR}/runs"
cfg["paths"]["reports_dir"] = f"{OUTPUT_DIR}/reports"
cfg["paths"]["models_dir"] = f"{OUTPUT_DIR}/models"
cfg["training"]["export_weights_to"] = f"{OUTPUT_DIR}/models/road_damage_yolo.pt"
cfg["inference"]["weights"] = f"{OUTPUT_DIR}/models/road_damage_yolo.pt"
cfg["training"]["workers"] = min(cfg["training"]["workers"], os.cpu_count() or 2)
cfg["dataset"]["extra_dirs"] = [str(EXTRA_DIR)] if EXTRA_DATASET else []  # training-only extra images
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Outputs will be saved to", OUTPUT_DIR)

## Copy / extract the dataset

In [ ]:
# Copy/extract to Colab's fast local disk (reading thousands of small files from Drive is slow).
import shutil, urllib.request, zipfile

def prepare(source, target, is_url=False):
    marker = target / ".ready"
    if marker.exists():
        print("Already prepared in", target)
        return
    if is_url:
        download = Path("/content/dataset_download.zip")
        print("Downloading", source, "...")
        urllib.request.urlretrieve(source, download)
        print(f"Downloaded {download.stat().st_size / 1e6:.0f} MB")
        if not zipfile.is_zipfile(download):
            raise ValueError("The link did not return a zip file. Use a DIRECT download link to the zip.")
        source = download
    source = Path(source)
    if source.is_dir():
        print("Copying folder", source, "... (can take 10+ minutes for thousands of files)")
        shutil.copytree(source, target, dirs_exist_ok=True)
    else:
        size_mb = source.stat().st_size / 1e6
        if not zipfile.is_zipfile(source):
            raise ValueError(f"{source} ({size_mb:.0f} MB) is not a complete zip file. If you uploaded it via the "
                             "Files panel, wait until the upload has finished (progress indicator at the bottom of "
                             "the Files panel), then run this cell again.")
        print(f"Extracting {source} ({size_mb:.0f} MB) ... (a few minutes)")
        with zipfile.ZipFile(source) as zf:
            zf.extractall(target)
    marker.touch()
    print("Done:", target)

prepare(DATASET, RAW_DIR, IS_URL)
if EXTRA_DATASET:
    prepare(EXTRA_DATASET, EXTRA_DIR)
    n_xml = sum(1 for _ in EXTRA_DIR.rglob("*.xml"))
    print(f"Extra images with XML labels: {n_xml}")
    if n_xml == 0:
        raise ValueError("EXTRA_DATASET contains no XML files - label the images first (src/annotate_boxes.py).")

## Install Ultralytics
Colab already has a GPU build of PyTorch.

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3.0"])
import ultralytics
print("Ultralytics", ultralytics.__version__)

## 1. Inspect the dataset
Check that *Matched image/XML pairs* is in the thousands and all four classes have counts.

In [ ]:
run([sys.executable, "src/inspect_dataset.py", "--raw-dir", RAW_DIR])

## 2. Convert XML → YOLO labels

In [ ]:
run([sys.executable, "src/convert_voc_to_yolo.py", "--raw-dir", RAW_DIR, "--force"])

## 3. Split into train / val / test and write data.yaml
The split is seeded, so it is identical every session (needed for resuming).

In [ ]:
run([sys.executable, "src/split_dataset.py", "--force"])

## 4. Train
Roughly 2-4 hours on a T4 with the defaults (early stopping may end it sooner). Keep this browser tab open.

In [ ]:
train_args = [sys.executable, "src/train_detector.py"]
if RESUME:
    train_args += ["--resume"]
elif QUICK_TEST:
    train_args += ["--epochs", 1, "--name", "quick_test"]
else:
    if EPOCHS:
        train_args += ["--epochs", EPOCHS]
    for flag, value in (("--model", MODEL), ("--imgsz", IMGSZ), ("--batch", BATCH)):
        if value:
            train_args += [flag, value]
run(train_args)

## 5. Evaluate on the test split

In [ ]:
run([sys.executable, "src/evaluate_detector.py"])

## 6. Done

In [ ]:
model_path = Path(OUTPUT_DIR) / "models" / "road_damage_yolo.pt"
print("Trained model :", model_path, "(exists)" if model_path.exists() else "(MISSING)")
print("Reports       :", Path(OUTPUT_DIR) / "reports" / "evaluation")
print("Training runs :", Path(OUTPUT_DIR) / "runs")
print("\nDownload road_damage_yolo.pt from Google Drive (My Drive/road-damage/outputs/models/)")
print("and copy it into your local project's models/ folder, then run: streamlit run app.py")